In [ ]:
!pip install numpy pandas matplotlib seaborn scikit-learn
!pip install lightgbm
!pip install shap
!pip install joblib
!pip install ipywidgets
!pip install tqdm


In [ ]:
# 1. quick status
!nvidia-smi
!pwd
!rm -rf /content/LightGBM

# 2. install build deps (boost dev, cmake, gcc, python dev)
!sudo apt-get update -y
!sudo apt-get install -y build-essential cmake git python3-dev \
    libboost-all-dev libboost-filesystem-dev libboost-system-dev \
    wget unzip

# 3. clone fresh repo
!git clone --recursive https://github.com/microsoft/LightGBM /content/LightGBM

# 4. create clean build dir
%cd /content/LightGBM
!rm -rf build
!mkdir -p build
%cd build

# 5. configure cmake: point explicitly at system Boost + CUDA.
#    Add Boost_NO_BOOST_CMAKE=ON to prefer the classic FindBoost behaviour.
!cmake -DBOOST_ROOT=/usr \
       -DBOOST_INCLUDEDIR=/usr/include \
       -DBOOST_LIBRARYDIR=/usr/lib/x86_64-linux-gnu \
       -DBoost_NO_BOOST_CMAKE=ON \
       -DUSE_GPU=1 -DUSE_CUDA=1 \
       -DCUDA_TOOLKIT_ROOT_DIR=/usr/local/cuda \
       -DCMAKE_BUILD_TYPE=Release ..

# 6. build
!make -j$(nproc)

# 7. install python package
%cd /content/LightGBM/python-package
!python3 setup.py install

# 8. verify (tiny GPU train)
!python3 - << 'PY'
import sys
try:
    import lightgbm as lgb, numpy as np
    print("lightgbm:", getattr(lgb, '__version__', 'unknown'))
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=200, n_features=10, random_state=0)
    dtrain = lgb.Dataset(X, label=y)
    params = {"objective":"binary","device":"gpu","verbose":-1}
    lgb.train(params, dtrain, num_boost_round=5)
    print("CUDA GPU train OK ✔")
except Exception as e:
    print("GPU verification failed:", e)
    sys.exit(2)
PY


In [ ]:
# Python cell
import lightgbm as lgb
import numpy as np
from sklearn.datasets import make_classification
print("lightgbm version:", getattr(lgb, "__version__", "unknown"))

X, y = make_classification(n_samples=300, n_features=12, random_state=0)
dtrain = lgb.Dataset(X, label=y)
params = {"objective": "binary", "device": "gpu", "verbose": -1}
try:
    bst = lgb.train(params, dtrain, num_boost_round=10)
    print("GPU LightGBM tiny train: OK ✅")
except Exception as e:
    print("GPU LightGBM tiny train: FAILED ❌")
    print("Error:", e)


In [ ]:
%cd /content
!pwd
!ls -la


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# install common Python libs used by your script (keeps LightGBM as-is)
!pip install --upgrade pip setuptools wheel
!pip install numpy pandas matplotlib seaborn scikit-learn shap joblib


In [ ]:
%cd /content
!pwd


In [ ]:
# PYTHON cell
import os
csv_path = "/content/drive/MyDrive/Aganitha/Model/RFF/AB_allebmpcdesc_clean.csv"
out_dir = "/content/drive/MyDrive/Aganitha/Model/RFF/all_output"
print("CSV exists:", os.path.exists(csv_path))
print("Output dir exists:", os.path.exists(out_dir))
if not os.path.exists(out_dir):
    os.makedirs(out_dir, exist_ok=True)
    print("Created output dir:", out_dir)


In [ ]:
ls -la "/content/drive/MyDrive/Aganitha/Model/RFF"

In [ ]:
ls -lb "/content/drive/MyDrive/Aganitha/Model/RFF"

In [ ]:
!ls -la "/content/drive/MyDrive/Aganitha/Model/RFF"
!ls -lb "/content/drive/MyDrive/Aganitha/Model/RFF"

In [ ]:
from google.colab import drive
drive.flush_and_unmount()



In [ ]:
!sudo rm -rf /content/drive
!mkdir -p /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!ls -la /content/drive


In [ ]:
!ls -la "/content/drive/MyDrive/Aganitha/Model/RFF"


In [ ]:
from google.colab import files
uploaded = files.upload()

# move uploaded file into your proper Drive folder
import shutil
for fn in uploaded:
    shutil.move(fn, f"/content/drive/MyDrive/Aganitha/Model/RFF/{fn}")
    print("Moved:", fn)


In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive/Aganitha/Model/RFF/AB_allebmpcdesc_clean_top15_afterlogloasso.csv"))


This is L1 lasso for selectiono fhte features

In [ ]:
# l1_select_with_label_and_exclusions.py
import json
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
import matplotlib.pyplot as plt

# ---------------- USER CONFIG (no path changes) ----------------
CSV_PATH = "/content/drive/MyDrive/Aganitha/Model/RFF/descriptors_batch_all_merged.csv"
OUT_DIR = Path("/content/drive/MyDrive/Aganitha/Model/RFF/L1_new3_with_label")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
TOP_K = 20
MIN_NONNA_RATIO = 0.3   # drop features with < 30% non-NA
DEVELOPABLE_FRACTION_CUTOFF = 0.95  # your rule

# ---------------- WEIGHTS and THRESHOLDS (provided) ----------------
WEIGHTS = {
    "normalized_titer_productionbatch1_avg": 1.0,
    "titer_productionbatch1_avg": 1.0,
    "purity_%lc+hc_avg": 1.0,
    "sec_%monomer_avg": 1.2,
    "tonset_nanodsf_avg": 1.0,
    "tm1_nanodsf_avg": 1.0,
    "tm2_nanodsf_avg": 1.0,
    "smac_rt_avg": 1.2,
    "hic_rt_avg": 1.2,
    "hac_rt_avg": 1.2,
    "acsins_dLmax_ph7.4_avg": 1.2,
    "acsins_dLmax_ph6.0_avg": 1.2,
    "polyreactivity_prscore_cho_avg": 1.2,
    "polyreactivity_prscore_ova_avg": 1.2,
}

THRESHOLDS = {
    "normalized_titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "purity_%lc+hc_avg": {"high": 95.0, "medium": 90.0, "direction": "high"},
    "sec_%monomer_avg": {"high": 97.0, "medium": 95.0, "direction": "high"},
    "tonset_nanodsf_avg": {"high": 60.0, "medium": 55.0, "direction": "high"},
    "tm1_nanodsf_avg": {"high": 63.0, "medium": 60.0, "direction": "high"},
    "tm2_nanodsf_avg": {"high": 75.0, "medium": 70.0, "direction": "high"},
    "smac_rt_avg": {"high": 12.8, "medium": 15.0, "direction": "low"},
    "hic_rt_avg": {"high": 11.7, "medium": 14.0, "direction": "low"},
    "hac_rt_avg": {"high": 12.0, "medium": 15.0, "direction": "low"},
    "acsins_dLmax_ph7.4_avg": {"high": 11.8, "medium": 20.0, "direction": "low"},
    "acsins_dLmax_ph6.0_avg": {"high": 12.0, "medium": 20.0, "direction": "low"},
    "polyreactivity_prscore_cho_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
    "polyreactivity_prscore_ova_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
}

# ---------------- Exact excludes (user insisted) ----------------
EXCLUDE_COLS = [
    "antibody_id", "antibody_name", "hc_subtype", "lc_subtype",
    "highest_clinical_trial_asof_feb2025", "est_status_asof_feb2025",
    "SEQ_VH_clean", "SEQ_VL_clean", "SEQ_VHVL_concat_linker",
    "SEQ_HC_full_clean", "SEQ_LC_full_clean"
]
EXTRA_EXCLUDE = {"titer_productionbatch1_avg", "hac_rt_avg"}  # enforced
DERIVED_LABEL_COLS = {
    "developable_bin", "developability_fraction",
    "weighted_pass_sum", "available_weight_sum", "developability_index_unweighted",
    "developability_index", "developability_score"
}

EXCLUDE_PATTERNS = [
    "_pass", "_avg_pass", "_binary", "_flag", "pass_", "pass$", "_percent_pass"
]

# ---------------- Load data ----------------
df = pd.read_csv(CSV_PATH)
print("Loaded:", CSV_PATH, "shape:", df.shape)

# ---------------- Build label (developable_bin) from THRESHOLDS + WEIGHTS ----------------
# For each thresholded assay we compute score = weight * (1 if high-pass, 0.5 if medium-pass, 0 if fail)
assay_keys = list(THRESHOLDS.keys())
weights_map = WEIGHTS.copy()

# Ensure the weights cover all thresholds keys
for k in assay_keys:
    if k not in weights_map:
        weights_map[k] = 1.0

# Create per-assay pass scores (use numeric values; missing -> treat as NaN and skip from available weight)
weighted_scores = []
available_weights = []

for assay in assay_keys:
    if assay not in df.columns:
        # try common variations (e.g. without suffix); skip if absent
        print(f"WARNING: assay column {assay} not found in CSV; it will be skipped from label calc.")
        continue
    thr = THRESHOLDS[assay]
    direction = thr["direction"]
    high = thr["high"]
    medium = thr["medium"]
    w = weights_map.get(assay, 1.0)

    col = df[assay]
    # vectorized scoring: set 1 for high, 0.5 for medium, 0 for fail, NaN stays NaN
    score_vec = pd.Series(index=df.index, dtype=float)

    if direction == "high":
        score_vec = np.where(col >= high, 1.0, np.where(col >= medium, 0.5, 0.0))
    else:  # direction == "low" (lower values are better)
        score_vec = np.where(col <= high, 1.0, np.where(col <= medium, 0.5, 0.0))

    # convert to numeric series and multiply by weight; treat NaNs properly
    score_series = pd.Series(score_vec, index=df.index).astype(float) * float(w)
    # mark available weight only where original value notna
    avail_series = pd.Series(np.where(col.notna(), float(w), 0.0), index=df.index)

    weighted_scores.append(score_series)
    available_weights.append(avail_series)

# If none of the assays were found, abort
if len(weighted_scores) == 0:
    raise RuntimeError("No assay columns from THRESHOLDS were found in CSV to build developable_bin.")

# Sum weighted scores and available weights
weighted_pass_sum = pd.concat(weighted_scores, axis=1).sum(axis=1)
available_weight_sum = pd.concat(available_weights, axis=1).sum(axis=1)

# compute developability_fraction where available_weight_sum > 0
with np.errstate(divide='ignore', invalid='ignore'):
    developability_fraction = np.where(available_weight_sum > 0, weighted_pass_sum / available_weight_sum, np.nan)

df["developability_fraction"] = developability_fraction
df["weighted_pass_sum"] = weighted_pass_sum
df["available_weight_sum"] = available_weight_sum

# derive developable_bin according to your rule
df["developable_bin"] = df["developability_fraction"].apply(lambda v: int(v >= DEVELOPABLE_FRACTION_CUTOFF) if not pd.isna(v) else np.nan)
print("Derived developable_bin; value counts (including NaN):\n", df["developable_bin"].value_counts(dropna=False))

# keep only rows where label exists
df = df[df["developable_bin"].notna()].copy()
n_rows = len(df)
print("Rows with label available:", n_rows)

# ---------------- Build exclude set and candidate features ----------------
exclude_set = set(THRESHOLDS.keys()) | set(EXCLUDE_COLS) | EXTRA_EXCLUDE | DERIVED_LABEL_COLS
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Total numeric columns:", len(numeric_cols))

def is_excluded_col(col):
    if col in exclude_set:
        return True
    # pattern based exclusion
    low = col.lower()
    for pat in EXCLUDE_PATTERNS:
        # treat patterns as substrings; patterns provided intentionally simple
        if pat.replace("$", "") in low:
            return True
    # exclude columns that contain any of the threshold keys as substrings (safe)
    for t in THRESHOLDS.keys():
        if t in col and col != t:
            return True
    # exclude columns that include 'pass' (safety)
    if "pass" in low:
        return True
    # exclude derived label names just in case
    if col in DERIVED_LABEL_COLS or col in {"developable_bin", "developability_fraction"}:
        return True
    return False

candidate_features = []
for c in numeric_cols:
    if is_excluded_col(c):
        continue
    nonna_ratio = df[c].notna().sum() / float(n_rows)
    if nonna_ratio < MIN_NONNA_RATIO:
        continue
    candidate_features.append(c)

print("Candidate features after exclusions:", len(candidate_features))
print("Example candidates (first 40):", candidate_features[:40])

if len(candidate_features) == 0:
    raise RuntimeError("No candidate features after exclusions. Adjust MIN_NONNA_RATIO or exclusions.")

# ---------------- Prepare X, y and split ----------------
X = df[candidate_features].copy()
y = df["developable_bin"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print("Train/test shapes:", X_train.shape, X_test.shape)

# ---------------- L1 selection pipeline ----------------
preproc = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
pipe = Pipeline([("preproc", preproc), ("clf", LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000, random_state=RANDOM_STATE))])

param_grid = {"clf__C": C_GRID}
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring="f1", n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("GridSearch best params:", grid.best_params_, " Best CV F1:", grid.best_score_)

# ---------------- Extract ranking and save ----------------
best_pipe = grid.best_estimator_
coefs = best_pipe.named_steps["clf"].coef_.ravel()

feat_df = pd.DataFrame({"feature": candidate_features, "coef": coefs, "abs_coef": np.abs(coefs)})
feat_df = feat_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)

feat_df.to_csv(OUT_DIR / "l1_feature_ranking_full.csv", index=False)
top_k = feat_df.head(TOP_K)
top_k.to_csv(OUT_DIR / f"l1_top_{TOP_K}.csv", index=False)
nonzero = feat_df[feat_df["abs_coef"] > 1e-8].copy()
nonzero.to_csv(OUT_DIR / "l1_nonzero_selected_features.csv", index=False)

# ---------------- Plot top K coefficients ----------------
plot_df = top_k.iloc[::-1]
plt.figure(figsize=(8, max(4, 0.3*len(plot_df))))
plt.barh(plot_df["feature"], plot_df["coef"])
plt.xlabel("Coefficient (signed)")
plt.title(f"L1 coefficients (top {len(plot_df)}) — C={grid.best_params_['clf__C']}")
plt.tight_layout()
plt.savefig(OUT_DIR / f"l1_coefficients_barh_top{TOP_K}.png", dpi=150)
plt.close()

# ---------------- Save summary ----------------
summary = {
    "csv_path": CSV_PATH,
    "n_rows_with_label": int(n_rows),
    "n_numeric_cols": int(len(numeric_cols)),
    "n_candidate_features": int(len(candidate_features)),
    "top_k": TOP_K,
    "best_C": grid.best_params_["clf__C"],
    "cv_f1": float(grid.best_score_)
}
with open(OUT_DIR / "l1_selection_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Done. Outputs in:", OUT_DIR)


L1 training and L2 Eval

In [ ]:
# l1_selector_then_eval_l2.py
import json
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss,
    roc_curve, auc, log_loss, mean_squared_error, r2_score, precision_recall_curve
)
import matplotlib.pyplot as plt

# ---------------- USER CONFIG (no path changes) ----------------
CSV_PATH = "/content/drive/MyDrive/Aganitha/Model/RFF/descriptors_batch_all_merged.csv"
OUT_DIR = Path("/content/drive/MyDrive/Aganitha/Model/RFF/L1_new4_eval")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
C_GRID_L1 = [0.001, 0.01, 0.1, 1.0, 10.0]   # for L1 selection
C_GRID_L2 = [0.01, 0.1, 1.0, 10.0]         # for L2 evaluation model
TOP_K = 20
MIN_NONNA_RATIO = 0.3
DEVELOPABLE_FRACTION_CUTOFF = 0.95

# ---------------- WEIGHTS and THRESHOLDS (provided) ----------------
WEIGHTS = {
    "normalized_titer_productionbatch1_avg": 1.0,
    "titer_productionbatch1_avg": 1.0,
    "purity_%lc+hc_avg": 1.0,
    "sec_%monomer_avg": 1.2,
    "tonset_nanodsf_avg": 1.0,
    "tm1_nanodsf_avg": 1.0,
    "tm2_nanodsf_avg": 1.0,
    "smac_rt_avg": 1.2,
    "hic_rt_avg": 1.2,
    "hac_rt_avg": 1.2,
    "acsins_dLmax_ph7.4_avg": 1.2,
    "acsins_dLmax_ph6.0_avg": 1.2,
    "polyreactivity_prscore_cho_avg": 1.2,
    "polyreactivity_prscore_ova_avg": 1.2,
}

THRESHOLDS = {
    "normalized_titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "purity_%lc+hc_avg": {"high": 95.0, "medium": 90.0, "direction": "high"},
    "sec_%monomer_avg": {"high": 97.0, "medium": 95.0, "direction": "high"},
    "tonset_nanodsf_avg": {"high": 60.0, "medium": 55.0, "direction": "high"},
    "tm1_nanodsf_avg": {"high": 63.0, "medium": 60.0, "direction": "high"},
    "tm2_nanodsf_avg": {"high": 75.0, "medium": 70.0, "direction": "high"},
    "smac_rt_avg": {"high": 12.8, "medium": 15.0, "direction": "low"},
    "hic_rt_avg": {"high": 11.7, "medium": 14.0, "direction": "low"},
    "hac_rt_avg": {"high": 12.0, "medium": 15.0, "direction": "low"},
    "acsins_dLmax_ph7.4_avg": {"high": 11.8, "medium": 20.0, "direction": "low"},
    "acsins_dLmax_ph6.0_avg": {"high": 12.0, "medium": 20.0, "direction": "low"},
    "polyreactivity_prscore_cho_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
    "polyreactivity_prscore_ova_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
}

# ---------------- Exact excludes (user insisted) ----------------
EXCLUDE_COLS = [
    "antibody_id", "antibody_name", "hc_subtype", "lc_subtype",
    "highest_clinical_trial_asof_feb2025", "est_status_asof_feb2025",
    "SEQ_VH_clean", "SEQ_VL_clean", "SEQ_VHVL_concat_linker",
    "SEQ_HC_full_clean", "SEQ_LC_full_clean"
]
EXTRA_EXCLUDE = {"titer_productionbatch1_avg", "hac_rt_avg"}  # enforced
DERIVED_LABEL_COLS = {
    "developable_bin", "developability_fraction",
    "weighted_pass_sum", "available_weight_sum", "developability_index_unweighted",
    "developability_index", "developability_score"
}
EXCLUDE_PATTERNS = ["_pass", "_avg_pass", "_binary", "_flag", "pass_", "pass$", "_percent_pass"]

# ---------------- Load ----------------
df = pd.read_csv(CSV_PATH)
print("Loaded:", CSV_PATH, "shape:", df.shape)

# ---------------- Build label ----------------
assay_keys = list(THRESHOLDS.keys())
weights_map = WEIGHTS.copy()
for k in assay_keys:
    if k not in weights_map:
        weights_map[k] = 1.0

weighted_scores = []
available_weights = []
found_any = False

for assay in assay_keys:
    if assay not in df.columns:
        print(f"WARNING: assay column {assay} not found; skipping it for label creation.")
        continue
    found_any = True
    thr = THRESHOLDS[assay]
    direction = thr["direction"]
    high = thr["high"]
    medium = thr["medium"]
    w = float(weights_map.get(assay, 1.0))
    col = df[assay]

    if direction == "high":
        score_vec = np.where(col >= high, 1.0, np.where(col >= medium, 0.5, 0.0))
    else:
        score_vec = np.where(col <= high, 1.0, np.where(col <= medium, 0.5, 0.0))

    score_series = pd.Series(score_vec, index=df.index).astype(float) * w
    avail_series = pd.Series(np.where(col.notna(), w, 0.0), index=df.index)

    weighted_scores.append(score_series)
    available_weights.append(avail_series)

if not found_any:
    raise RuntimeError("No assay columns found to construct developable_bin — aborting.")

weighted_pass_sum = pd.concat(weighted_scores, axis=1).sum(axis=1)
available_weight_sum = pd.concat(available_weights, axis=1).sum(axis=1)
with np.errstate(divide='ignore', invalid='ignore'):
    developability_fraction = np.where(available_weight_sum > 0, weighted_pass_sum / available_weight_sum, np.nan)

df["developability_fraction"] = developability_fraction
df["weighted_pass_sum"] = weighted_pass_sum
df["available_weight_sum"] = available_weight_sum
df["developable_bin"] = pd.Series(np.where(df["developability_fraction"] >= DEVELOPABLE_FRACTION_CUTOFF, 1, 0), index=df.index)
print("Derived developable_bin counts (including NaNs handled):\n", df["developable_bin"].value_counts(dropna=False))

# keep only rows where label exists
df = df[df["available_weight_sum"] > 0].copy()
n_rows = len(df)
print("Rows used:", n_rows)

# ---------------- Build candidate features (exclude assays etc) ----------------
exclude_set = set(THRESHOLDS.keys()) | set(EXCLUDE_COLS) | EXTRA_EXCLUDE | DERIVED_LABEL_COLS
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

def is_excluded_col(col):
    if col in exclude_set:
        return True
    low = col.lower()
    for pat in EXCLUDE_PATTERNS:
        if pat.replace("$","") in low:
            return True
    for t in THRESHOLDS.keys():
        if t in col and col != t:
            return True
    if "pass" in low:
        return True
    if col in DERIVED_LABEL_COLS or col in {"developable_bin", "developability_fraction"}:
        return True
    return False

candidate_features = []
for c in numeric_cols:
    if is_excluded_col(c):
        continue
    nonna_ratio = df[c].notna().sum() / float(n_rows)
    if nonna_ratio < MIN_NONNA_RATIO:
        continue
    candidate_features.append(c)

print("Candidate numeric features count:", len(candidate_features))
print("Example candidates (first 40):", candidate_features[:40])

if len(candidate_features) == 0:
    raise RuntimeError("No candidate features after exclusion. Adjust MIN_NONNA_RATIO or exclusions.")

# ---------------- Prepare X,y and split ----------------
X = df[candidate_features].copy()
y = df["developable_bin"].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
print("Train/test shapes:", X_train.shape, X_test.shape)

# ---------------- L1 selection on TRAIN only ----------------
preproc_sel = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
pipe_l1 = Pipeline([("preproc", preproc_sel), ("clf", LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000, random_state=RANDOM_STATE))])
param_grid_l1 = {"clf__C": C_GRID_L1}
cv_sel = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
grid_l1 = GridSearchCV(pipe_l1, param_grid=param_grid_l1, cv=cv_sel, scoring="f1", n_jobs=-1, verbose=1)
grid_l1.fit(X_train, y_train)

print("L1 selection best params:", grid_l1.best_params_, "best CV F1:", grid_l1.best_score_)

best_l1 = grid_l1.best_estimator_
coefs = best_l1.named_steps["clf"].coef_.ravel()
feat_rank_df = pd.DataFrame({"feature": candidate_features, "coef": coefs, "abs_coef": np.abs(coefs)})
feat_rank_df = feat_rank_df.sort_values("abs_coef", ascending=False).reset_index(drop=True)
feat_rank_df.to_csv(OUT_DIR / "l1_feature_ranking_full.csv", index=False)
feat_rank_df.head(TOP_K).to_csv(OUT_DIR / f"l1_top_{TOP_K}.csv", index=False)
feat_rank_df[feat_rank_df["abs_coef"] > 1e-8].to_csv(OUT_DIR / "l1_nonzero_selected_features.csv", index=False)
print("Saved L1 ranking and selected lists.")

# ---------------- Train an evaluation model (L2 Logistic) on selected features (train) ----------------
selected_nonzero = feat_rank_df[feat_rank_df["abs_coef"] > 1e-8]["feature"].tolist()
# If none non-zero, fall back to top-K
if len(selected_nonzero) == 0:
    selected_nonzero = feat_rank_df.head(TOP_K)["feature"].tolist()
print("Selected features used for evaluation (count):", len(selected_nonzero))

X_train_sel = X_train[selected_nonzero].copy()
X_test_sel = X_test[selected_nonzero].copy()

# L2 grid on selected features
preproc_eval = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
pipe_l2 = Pipeline([("preproc", preproc_eval), ("clf", LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000, random_state=RANDOM_STATE))])
param_grid_l2 = {"clf__C": C_GRID_L2}
cv_eval = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
grid_l2 = GridSearchCV(pipe_l2, param_grid=param_grid_l2, cv=cv_eval, scoring="f1", n_jobs=-1, verbose=1)
grid_l2.fit(X_train_sel, y_train)

best_l2 = grid_l2.best_estimator_
print("L2 eval best params:", grid_l2.best_params_, "best CV F1:", grid_l2.best_score_)

# ---------------- Predictions on test ----------------
y_prob = best_l2.predict_proba(X_test_sel)[:,1]
y_pred = best_l2.predict(X_test_sel)

# Basic metrics
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
    "pr_auc": float(average_precision_score(y_test, y_prob)),
    "brier_loss": float(brier_score_loss(y_test, y_prob)),
    "logloss": float(log_loss(y_test, y_prob)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
}
with open(OUT_DIR / "evaluation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved evaluation metrics:", metrics)

# ---------------- Extended metrics: R2 / RMSE / top20 ----------------
r2 = float(r2_score(y_test, y_prob))
mse = mean_squared_error(y_test, y_prob)
rmse = float(np.sqrt(mse))

def top_k_metrics(y_true, y_prob, k=20):
    n = len(y_prob)
    if n == 0:
        return {"r2_topk": None, "rmse_topk": None}
    idx = np.argsort(-y_prob)
    k2 = min(k, n)
    top_idx = idx[:k2]
    r2_top = float(r2_score(y_true.iloc[top_idx], y_prob[top_idx])) if k2>0 else None
    mse_top = mean_squared_error(y_true.iloc[top_idx], y_prob[top_idx])
    rmse_top = float(np.sqrt(mse_top)) if k2>0 else None
    return {"r2_topk": r2_top, "rmse_topk": rmse_top}

top20 = top_k_metrics(y_test.reset_index(drop=True), y_prob, k=20)

extended = {"r2": r2, "rmse": rmse, "top20": top20}
with open(OUT_DIR / "evaluation_extended_metrics.json", "w") as f:
    json.dump(extended, f, indent=2)

# ---------------- Plots ----------------
# ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(7,6))
plt.plot(fpr, tpr, label=f"L2 (AUC = {roc_auc:.3f})")
plt.plot([0,1],[0,1],'--', linewidth=0.7)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Eval model on selected features)")
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', linewidth=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "roc_curve_eval.png", dpi=150)
plt.close()

# PR curve
prec, rec, _ = precision_recall_curve(y_test, y_prob)
plt.figure(figsize=(7,6))
plt.plot(rec, prec, label=f"PR (AP = {metrics['pr_auc']:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (Eval model)")
plt.legend(loc="lower left")
plt.grid(True, linestyle=':', linewidth=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "pr_curve_eval.png", dpi=150)
plt.close()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="viridis")
for (i,j), val in np.ndenumerate(cm):
    ax.text(j, i, val, ha="center", va="center", color="w", fontsize=12)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Eval model)")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_eval.png", dpi=150)
plt.close()

# Metric comparison bar
labels = ["Accuracy", "F1", "Precision", "Recall", "ROC AUC", "PR AUC", "Brier", "LogLoss"]
vals = [metrics["accuracy"], metrics["f1"], metrics["precision"], metrics["recall"], metrics["roc_auc"], metrics["pr_auc"], metrics["brier_loss"], metrics["logloss"]]
x = np.arange(len(labels))
width = 0.6
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x, vals, width)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_title("Evaluation Metrics (selected features)")
plt.tight_layout()
plt.savefig(OUT_DIR / "metrics_comparison_eval.png", dpi=150)
plt.close()

# Calibration scatter: prob vs true (jitter)
plt.figure(figsize=(6,4))
plt.scatter(y_prob, y_test + (np.random.randn(len(y_test)) * 0.02), alpha=0.7, s=20)
plt.xlabel("Predicted probability")
plt.ylabel("True label (jittered)")
plt.title("Predicted prob vs true label (jitter)")
plt.tight_layout()
plt.savefig(OUT_DIR / "prob_vs_true_scatter_eval.png", dpi=150)
plt.close()

# Learning curve (train sizes vs CV F1 for eval model)
train_sizes, train_scores, test_scores = learning_curve(
    grid_l2.best_estimator_, X_train_sel, y_train, cv=cv_eval, scoring="f1",
    train_sizes=np.linspace(0.2,1.0,5), n_jobs=-1, random_state=RANDOM_STATE
)
train_scores_mean = np.mean(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
plt.figure(figsize=(7,5))
plt.plot(train_sizes, train_scores_mean, "o-", label="Train F1")
plt.plot(train_sizes, test_scores_mean, "o-", label="CV F1")
plt.xlabel("Training examples")
plt.ylabel("F1 score")
plt.title("Learning Curve (Eval model)")
plt.legend()
plt.grid(True, linestyle=':', linewidth=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "learning_curve_eval.png", dpi=150)
plt.close()

# Save predictions with probs
preds_df = pd.DataFrame({
    "index": X_test_sel.index,
    "developable_true": y_test.values,
    "prob": y_prob,
    "pred": y_pred
})
preds_df.to_csv(OUT_DIR / "eval_test_predictions.csv", index=False)

# Save summary
summary = {
    "selected_features_count": len(selected_nonzero),
    "selected_features": selected_nonzero,
    "evaluation_metrics": metrics,
    "extended": extended,
    "l1_best_C": grid_l1.best_params_["clf__C"],
    "l2_best_C": grid_l2.best_params_["clf__C"]
}
with open(OUT_DIR / "l1_selection_eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("All diagnostics saved to:", OUT_DIR)


In [ ]:
#!/usr/bin/env python3
"""
Fast developability training — Stage-1 only (LightGBM RandomizedSearch)
Patched for speed on laptops: fewer CV folds, fewer iterations, no nested parallelism,
reduced n_estimators during search, timing, and optional sampling.
"""
import os
import json
import joblib
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# minimal sklearn imports
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, matthews_corrcoef, brier_score_loss
)

# LightGBM
import lightgbm as lgb

# quiet warnings / plot style
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_style("whitegrid")

# ---------------- CONFIG (patched) ----------------
INPUT_CSV = "/content/drive/MyDrive/Aganitha/Model/RFF/descriptors_batch_all_merged.csv"
BASE_OUTDIR = Path("/content/drive/MyDrive/Aganitha/Model/RFF/L1_new2")
RUN_PREFIX = "run"
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_TOP_FEATURES = 25

# ---- Speed tuning knobs ----
SAMPLE_FRAC = 1.0    # set to 0.5 for quick hyperparam search on half the data
SEARCH_N_ITER = 8    # reduced from 12 -> 8
SEARCH_CV_FOLDS = 3  # reduced from 5 -> 3
SEARCH_N_ESTIMATORS = [100, 200, 300]  # smaller during search; retrain final model later with more trees
# ---------------- end speed knobs ----------------

# weights / thresholds (unchanged logic)
WEIGHTS = {
    "normalized_titer_productionbatch1_avg": 1.0,
    "titer_productionbatch1_avg": 1.0,
    "purity_%lc+hc_avg": 1.0,
    "sec_%monomer_avg": 1.2,
    "tonset_nanodsf_avg": 1.0,
    "tm1_nanodsf_avg": 1.0,
    "tm2_nanodsf_avg": 1.0,
    "smac_rt_avg": 1.2,
    "hic_rt_avg": 1.2,
    "hac_rt_avg": 1.2,
    "acsins_dLmax_ph7.4_avg": 1.2,
    "acsins_dLmax_ph6.0_avg": 1.2,
    "polyreactivity_prscore_cho_avg": 1.2,
    "polyreactivity_prscore_ova_avg": 1.2,
}
TITER_WEIGHT = 0.35
WEIGHTS["normalized_titer_productionbatch1_avg"] = TITER_WEIGHT
WEIGHTS["titer_productionbatch1_avg"] = TITER_WEIGHT

DEVELOPABILITY_THRESHOLD = 0.98

EXCLUDE_COLS = [
    "antibody_id", "antibody_name", "hc_subtype", "lc_subtype",
    "highest_clinical_trial_asof_feb2025", "est_status_asof_feb2025",
    "SEQ_VH_clean", "SEQ_VL_clean", "SEQ_VHVL_concat_linker",
    "SEQ_HC_full_clean", "SEQ_LC_full_clean"
]

THRESHOLDS = {
    "normalized_titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high", "unit": "mg/L"},
    "titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high", "unit": "mg/L"},
    "purity_%lc+hc_avg": {"high": 95.0, "medium": 90.0, "direction": "high", "unit": "%"},
    "sec_%monomer_avg": {"high": 97.0, "medium": 95.0, "direction": "high", "unit": "%"},
    "tonset_nanodsf_avg": {"high": 60.0, "medium": 55.0, "direction": "high", "unit": "°C"},
    "tm1_nanodsf_avg": {"high": 63.0, "medium": 60.0, "direction": "high", "unit": "°C"},
    "tm2_nanodsf_avg": {"high": 75.0, "medium": 70.0, "direction": "high", "unit": "°C"},
    "smac_rt_avg": {"high": 12.8, "medium": 15.0, "direction": "low", "unit": "min"},
    "hic_rt_avg": {"high": 11.7, "medium": 14.0, "direction": "low", "unit": "min"},
    "hac_rt_avg": {"high": 12.0, "medium": 15.0, "direction": "low", "unit": "min"},
    "acsins_dLmax_ph7.4_avg": {"high": 11.8, "medium": 20.0, "direction": "low", "unit": "nm"},
    "acsins_dLmax_ph6.0_avg": {"high": 12.0, "medium": 20.0, "direction": "low", "unit": "nm"},
    "polyreactivity_prscore_cho_avg": {"high": 0.27, "medium": 0.35, "direction": "low", "unit": "PSR score"},
    "polyreactivity_prscore_ova_avg": {"high": 0.27, "medium": 0.35, "direction": "low", "unit": "PSR score"}
}

ASSAYS = list(THRESHOLDS.keys())
SOFT_ASSAYS = ["normalized_titer_productionbatch1_avg", "titer_productionbatch1_avg"]

# ---------------- helpers ----------------
def next_run_folder(base_dir: Path, prefix="run"):
    base_dir.mkdir(parents=True, exist_ok=True)
    existing = sorted([p.name for p in base_dir.iterdir() if p.is_dir() and p.name.startswith(prefix)])
    if not existing:
        return base_dir / f"{prefix}1"
    last = existing[-1]
    try:
        n = int(last.replace(prefix, ""))
        return base_dir / f"{prefix}{n+1}"
    except:
        i = 1
        while (base_dir / f"{prefix}{i}").exists():
            i += 1
        return base_dir / f"{prefix}{i}"

def safe_savefig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def tier_for_value(val, thr):
    if pd.isna(val): return np.nan
    d = thr["direction"]
    if d == "high":
        return "High" if val >= thr["high"] else ("Medium" if val >= thr["medium"] else "Low")
    if d == "low":
        return "High" if val <= thr["high"] else ("Medium" if val <= thr["medium"] else "Low")
    return np.nan

# ---------------- LOAD DATA ----------------
if not Path(INPUT_CSV).exists():
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)

# optional sampling for faster hyperparameter search
if SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE).reset_index(drop=True)

# ---------------- TIERS & PASSES ----------------
for assay, thr in THRESHOLDS.items():
    if assay not in df.columns:
        df[assay] = np.nan
    df[f"{assay}_tier"] = df[assay].apply(lambda v: tier_for_value(v, thr))
    df[f"{assay}_pass"] = df[f"{assay}_tier"].map({"High": 1, "Medium": 1, "Low": 0})

# ---------------- WEIGHTED DEVELOPABILITY ----------------
pass_cols = [f"{a}_pass" for a in ASSAYS]
weight_vec = np.array([WEIGHTS[a] for a in ASSAYS], dtype=float)
passes_matrix = df[pass_cols].to_numpy(dtype=float)
available_mask = ~np.isnan(passes_matrix)

weighted_pass = np.nansum(np.where(available_mask, passes_matrix * weight_vec, 0.0), axis=1)
available_weight = np.sum(np.where(available_mask, weight_vec, 0.0), axis=1)
weighted_fraction = np.where(available_weight > 0, weighted_pass / available_weight, np.nan)

df["weighted_pass_sum"] = weighted_pass
df["available_weight_sum"] = available_weight
df["developability_fraction"] = weighted_fraction
df["developability_index_unweighted"] = df[pass_cols].sum(axis=1, skipna=True)
df["developable_bin"] = (df["developability_fraction"] >= DEVELOPABILITY_THRESHOLD).astype(int)
df.loc[df["developability_fraction"].isna(), "developable_bin"] = np.nan

# ---------------- DIAGNOSTICS (console) ----------------
zero_mask = df["available_weight_sum"] == 0
print(f"Rows with NO assays available: {zero_mask.sum()}")
print("Developability fraction summary:")
print(df["developability_fraction"].describe())
print("Missingness per assay (fraction):")
print(df[ASSAYS].isna().mean().sort_values(ascending=False))

# ---------------- FEATURES FOR ML ----------------
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
tier_cols = [c for c in df.columns if c.endswith("_tier")]
pass_cols_actual = [c for c in df.columns if c.endswith("_pass")]

cols_to_remove = (
    set(tier_cols) |
    set(pass_cols_actual) |
    set(EXCLUDE_COLS) |
    set(ASSAYS) |
    {"developable_bin", "developability_fraction", "developability_index_unweighted",
     "weighted_pass_sum", "available_weight_sum"}
)

feature_cols = [c for c in numeric_cols if c not in cols_to_remove]
feat_missing_frac = df[feature_cols].isna().mean() if feature_cols else pd.Series(dtype=float)
feature_cols = [c for c in feature_cols if feat_missing_frac[c] <= 0.5]

print(f"LEAK-FREE FEATURE COUNT: {len(feature_cols)}")
if not feature_cols:
    raise RuntimeError("No valid features after removing assay leak. Add sequence-based features.")

imputer = SimpleImputer(strategy="median")
X_all = imputer.fit_transform(df[feature_cols].values)

y_series = df["developable_bin"]
has_label_mask = ~y_series.isna()
y_all = y_series[has_label_mask].astype(int).to_numpy()
X_all_labeled = X_all[has_label_mask.values, :]

stratify = y_all if len(np.unique(y_all)) > 1 else None
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all_labeled, y_all, df.index.values[has_label_mask.values],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=stratify
)

# ---------------- RUN DIR ----------------
run_dir = next_run_folder(BASE_OUTDIR, RUN_PREFIX)
run_dir.mkdir(parents=True, exist_ok=True)

config = {
    "input_file": INPUT_CSV,
    "date": datetime.utcnow().isoformat(),
    "thresholds": THRESHOLDS,
    "weights": WEIGHTS,
    "soft_assays": SOFT_ASSAYS,
    "developability_threshold_type": "weighted_fraction",
    "developability_threshold_value": DEVELOPABILITY_THRESHOLD,
    "n_features": len(feature_cols),
    "feature_columns": feature_cols
}
with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

with open(run_dir / "run_config.txt", "w") as f:
    f.write("=== STAGE-1 ONLY: RANDOMIZED SEARCH (LightGBM) ===\n")
    f.write(f"Run Directory: {run_dir}\nDate (UTC): {datetime.utcnow().isoformat()}\n")
    f.write(f"Features used: {len(feature_cols)}\n")

# ---------------- LIGHTGBM RANDOMIZED SEARCH (Stage-1 only) ----------------
print("Starting Stage-1 RandomizedSearchCV (LightGBM) — fast config")

# Reduced, focused search space (patched)
param_distributions_lgb = {
    "n_estimators": SEARCH_N_ESTIMATORS,
    "learning_rate": [0.02, 0.05, 0.1],
    "num_leaves": [15, 31],          # narrower
    "max_depth": [6, 8],
    "min_child_samples": [5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.4, 0.6],
    "reg_alpha": [0.0, 0.1],
    "reg_lambda": [0.0, 0.1],
    "class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=SEARCH_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# IMPORTANT: avoid nested parallelism — estimator single-threaded, RandomizedSearch parallelizes CV jobs
lgb_est = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1,       # single-thread inside estimator
    verbosity=-1
)

random_search = RandomizedSearchCV(
    estimator=lgb_est,
    param_distributions=param_distributions_lgb,
    n_iter=SEARCH_N_ITER,               # reduced iterations
    scoring="average_precision",
    cv=cv,
    refit=True,
    random_state=RANDOM_STATE,
    verbose=0,                          # silent in notebook
    n_jobs=-1                           # parallelize across CV folds
)

# timing wrapper
t0 = time.time()
random_search.fit(X_train, y_train)
elapsed = time.time() - t0
print(f"RandomizedSearchCV finished in {elapsed:.1f} s (~{elapsed/60:.1f} min)")

# collect results & best model
clf = random_search.best_estimator_
best_params = random_search.best_params_
config["algorithm"] = "LightGBM"
config["best_hyperparameters_lgb"] = best_params
config["best_cv_score_stage1_average_precision"] = float(random_search.best_score_)

# persist files
with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

pd.DataFrame(random_search.cv_results_).to_csv(run_dir / "cv_results_stage1.csv", index=False)
joblib.dump({"model": clf, "features": feature_cols, "imputer": imputer},
            str(run_dir / "lgb_stage1_model.joblib"))

print("Stage-1 complete. Best params:", best_params)
print("Best CV (average_precision):", random_search.best_score_)

# ---------------- PREDICTIONS & METRICS ----------------
y_proba = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)) if len(np.unique(y_test)) > 1 else float("nan"),
    "pr_auc": float(average_precision_score(y_test, y_proba)),
    "mcc": float(matthews_corrcoef(y_test, y_pred)) if len(np.unique(y_test)) > 1 else float("nan"),
    "brier_loss": float(brier_score_loss(y_test, y_proba)) if len(np.unique(y_test)) > 1 else float("nan")
}
tn, fp, fn, tp = 0, 0, 0, 0
try:
    cm = confusion_matrix(y_test, y_pred)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
except:
    pass
metrics.update({"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})

with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# ---------------- SAVE PREDICTIONS ----------------
preds_df = df.loc[idx_test, ["antibody_id", "antibody_name"] + ASSAYS].copy().reset_index(drop=True)
preds_df["developable_true"] = y_test
preds_df["developable_pred"] = y_pred
preds_df["developable_prob"] = y_proba
preds_df["developability_fraction"] = df.loc[idx_test, "developability_fraction"].values
preds_df["available_weight_sum"] = df.loc[idx_test, "available_weight_sum"].values
preds_df["weighted_pass_sum"] = df.loc[idx_test, "weighted_pass_sum"].values
preds_df.to_csv(run_dir / "predictions_test.csv", index=False)

# ---------------- PLOTS & feature importances (unchanged) ----------------
# ROC
if len(np.unique(y_test)) > 1:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    try:
        roc_auc = roc_auc_score(y_test, y_proba)
    except:
        roc_auc = np.nan
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0,1],[0,1], '--', color='gray')
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "roc_curve.png")

# PR Curve
if len(np.unique(y_test)) > 1:
    from sklearn.metrics import precision_recall_curve
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(recall_vals, precision_vals, label=f"AP = {ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "pr_curve.png")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4,3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
safe_savefig(plt.gcf(), run_dir / "confusion_matrix.png")

# Calibration
if len(np.unique(y_test)) > 1:
    from sklearn.calibration import calibration_curve as _cc
    prob_true, prob_pred = _cc(y_test, y_proba, n_bins=10)
    plt.figure(figsize=(6,5))
    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0,1],[0,1], '--', color='gray')
    plt.xlabel("Mean predicted probability"); plt.ylabel("Fraction of positives")
    plt.title("Calibration plot"); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "calibration.png")

# Feature Importances
fi = clf.feature_importances_
fi_idx = np.argsort(fi)[::-1][:N_TOP_FEATURES]
top_features = [(feature_cols[i], float(fi[i])) for i in fi_idx]
fi_df = pd.DataFrame(top_features, columns=["feature", "importance"])
fi_df.to_csv(run_dir / "feature_importances_top.csv", index=False)

plt.figure(figsize=(8, max(4, 0.25*len(fi_idx))))
sns.barplot(x=[f for _,f in top_features], y=[n for n,_ in top_features])
plt.xlabel("Importance"); plt.title("Top feature importances (LGBM)")
safe_savefig(plt.gcf(), run_dir / "feature_importances_top.png")

# Threshold sweep (unchanged)
sweep_thresholds = np.linspace(0.4, 0.9, 51)
sweep_results = []
mask = ~df["developable_bin"].isna()
if mask.sum() > 0:
    for thr in sweep_thresholds:
        bin_pred = (df["developability_fraction"] >= thr).astype(float)
        y_true_full = df.loc[mask, "developable_bin"].astype(int).to_numpy()
        y_pred_full = bin_pred.loc[mask].astype(int).to_numpy()
        sweep_results.append({
            "threshold": float(thr),
            "n": int(mask.sum()),
            "accuracy": float(accuracy_score(y_true_full, y_pred_full)),
            "precision": float(precision_score(y_true_full, y_pred_full, zero_division=0)),
            "recall": float(recall_score(y_true_full, y_pred_full, zero_division=0)),
            "f1": float(f1_score(y_true_full, y_pred_full, zero_division=0))
        })
pd.DataFrame(sweep_results).to_csv(run_dir / "threshold_sweep.csv", index=False)

# Final diagnostics
diag = {
    "n_total_rows": int(df.shape[0]),
    "n_rows_with_label": int(has_label_mask.sum()),
    "n_features_used": len(feature_cols),
    "developability_threshold_chosen": DEVELOPABILITY_THRESHOLD,
    "titer_weight_used": TITER_WEIGHT
}
with open(run_dir / "diagnostics.json", "w") as f:
    json.dump(diag, f, indent=2)

print(f"\nRUN COMPLETE → {run_dir}")
print("Stage-1 only run finished (fast config).")


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss,
    roc_curve, auc, log_loss, mean_squared_error, r2_score, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# === User-editable paths & settings ===
# Primary CSV path used previously (Colab / Drive)
CSV_PATH = "/content/drive/MyDrive/Aganitha/Model/RFF/all_246_antibodies_with_developability.csv"
# If you want to use the uploaded test CSV inside the notebook environment, uncomment below:
# CSV_PATH = "/mnt/data/Sequence derived features.csv"

OUT_DIR = Path("/content/drive/MyDrive/Aganitha/Model/RFF/logbase_lasso6")
OUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.20
TOP15_FEATURES = [
    "aa_frac_D", "PC_ESM_14", "PC_ProtBerty_30", "aa_frac_Q", "PC_ESM_49",
    "PC_ESM_20", "PC_ESM_3", "PC_ESM_16", "aa_frac_E", "PC_ESM_48",
    "PC_ProtBerty_8", "PC_ESM_23", "PC_ESM_1", "PC_ESM_34", "aliphatic_index"
]
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# === Load data ===
df = pd.read_csv(CSV_PATH)
print("Loaded dataset:", df.shape)

# Ensure label exists; derive if needed (uses developability_fraction and threshold 0.95 if present)
if "developable_bin" not in df.columns:
    if "developability_fraction" in df.columns:
        thr = 0.95
        df["developable_bin"] = df["developability_fraction"].apply(lambda v: int(v >= thr) if not pd.isna(v) else np.nan)
        print("Derived developable_bin from developability_fraction with threshold", thr)
    else:
        raise ValueError("No developable_bin or developability_fraction present in CSV.")

# Filter to rows with labels
df_labeled = df[df["developable_bin"].notna()].copy()
print("Labeled rows:", df_labeled.shape[0])

# Check features exist
missing_feats = [f for f in TOP15_FEATURES if f not in df_labeled.columns]
if missing_feats:
    raise KeyError(f"The following required features are missing from the CSV: {missing_feats}")

X = df_labeled[TOP15_FEATURES].copy()
y = df_labeled["developable_bin"].astype(int).copy()

# Train/test split (same logic as prior runs)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print("Train/test shapes:", X_train.shape, X_test.shape)

# === Pipeline: impute (median) -> scale -> logistic ===
base_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=5000))
])

# --- 1) L2 Logistic (baseline) ---
pipe_l2 = base_pipeline
# we'll tune inverse regularization C
param_grid_l2 = {
    "clf__penalty": ["l2"],
    "clf__solver": ["liblinear"],   # liblinear fine for small feature sets
    "clf__C": [0.01, 0.1, 1.0, 10.0]
}

grid_l2 = GridSearchCV(
    pipe_l2,
    param_grid=param_grid_l2,
    cv=CV,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)
grid_l2.fit(X_train, y_train)
best_l2 = grid_l2.best_estimator_
print("L2 best params:", grid_l2.best_params_)
print("L2 best CV score (F1):", grid_l2.best_score_)

# Evaluate on test set
y_prob_l2 = best_l2.predict_proba(X_test)[:,1]
y_pred_l2 = best_l2.predict(X_test)
metrics_l2 = {
    "accuracy": float(accuracy_score(y_test, y_pred_l2)),
    "precision": float(precision_score(y_test, y_pred_l2, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred_l2, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred_l2, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_prob_l2)),
    "pr_auc": float(average_precision_score(y_test, y_prob_l2)),
    "brier_loss": float(brier_score_loss(y_test, y_prob_l2)),
    "confusion_matrix": confusion_matrix(y_test, y_pred_l2).tolist()
}
print("L2 test metrics:", metrics_l2)

# Save L2 artifacts
pd.DataFrame({"feature": TOP15_FEATURES}).to_csv(OUT_DIR / "top15_features.csv", index=False)
with open(OUT_DIR / "metrics_l2.json", "w") as f:
    json.dump(metrics_l2, f, indent=2)

# --- 2) L1 Logistic (LASSO) for feature selection ---
pipe_l1 = base_pipeline
# L1 requires solvers that support L1 (liblinear is okay)
param_grid_l1 = {
    "clf__penalty": ["l1"],
    "clf__solver": ["liblinear"],
    "clf__C": [0.01, 0.1, 1.0, 10.0]
}

grid_l1 = GridSearchCV(
    pipe_l1,
    param_grid=param_grid_l1,
    cv=CV,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)
grid_l1.fit(X_train, y_train)
best_l1 = grid_l1.best_estimator_
print("L1 best params:", grid_l1.best_params_)
print("L1 best CV score (F1):", grid_l1.best_score_)

# Extract selected features (non-zero coefficients)
# get pipeline named step 'clf' coefficient after preprocessing
# We need to get the coef_ in the original feature order; StandardScaler doesn't change order
coefs = best_l1.named_steps["clf"].coef_.ravel()
selected_mask = np.abs(coefs) > 1e-8
selected_features = [f for f, sel in zip(TOP15_FEATURES, selected_mask) if sel]
print("Selected features by L1 (non-zero coef):", selected_features)

# Evaluate L1 on test
y_prob_l1 = best_l1.predict_proba(X_test)[:,1]
y_pred_l1 = best_l1.predict(X_test)
metrics_l1 = {
    "accuracy": float(accuracy_score(y_test, y_pred_l1)),
    "precision": float(precision_score(y_test, y_pred_l1, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred_l1, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred_l1, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_prob_l1)),
    "pr_auc": float(average_precision_score(y_test, y_prob_l1)),
    "brier_loss": float(brier_score_loss(y_test, y_prob_l1)),
    "confusion_matrix": confusion_matrix(y_test, y_pred_l1).tolist()
}
print("L1 test metrics:", metrics_l1)

# Save L1 artifacts
with open(OUT_DIR / "metrics_l1.json", "w") as f:
    json.dump(metrics_l1, f, indent=2)
pd.DataFrame({
    "feature": TOP15_FEATURES,
    "coef": coefs,
    "selected": selected_mask.astype(int)
}).to_csv(OUT_DIR / "l1_coeffs_and_selection.csv", index=False)
pd.DataFrame({"selected_features": selected_features}).to_csv(OUT_DIR / "l1_selected_features.csv", index=False)

# Save predictions (test)
preds_out = pd.DataFrame({
    "antibody_index": X_test.index,
    "developable_true": y_test,
    "prob_l2": y_prob_l2,
    "pred_l2": y_pred_l2,
    "prob_l1": y_prob_l1,
    "pred_l1": y_pred_l1
})
preds_out.to_csv(OUT_DIR / "logreg_test_predictions.csv", index=False)

# Summary output
summary = {
    "train_rows": int(X_train.shape[0]),
    "test_rows": int(X_test.shape[0]),
    "top15_features": TOP15_FEATURES,
    "l2_best_params": grid_l2.best_params_,
    "l1_best_params": grid_l1.best_params_,
    "metrics_l2": metrics_l2,
    "metrics_l1": metrics_l1,
    "l1_selected_features": selected_features
      }
with open(OUT_DIR / "logreg_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("All done. Artifacts written to:", OUT_DIR)
print("Key results summary:", summary)

# === ADDED PLOTTING/DIAGNOSTICS ===
# This section uses existing objects: best_l2, best_l1, X_test, y_test, y_prob_*, y_pred_*
# It will produce:
# - ROC curves (L2 vs L1)
# - Confusion matrices (if applicable)
# - Bar charts comparing Accuracy, F1, Precision, Recall, ROC AUC, PR AUC, Brier, LogLoss
# - R2 and RMSE computed on predicted probabilities vs true labels (overall + top-20 by prob)
# - Save plots to OUT_DIR

# safe helper to ensure numeric arrays
y_test_arr = np.asarray(y_test)

# roc curves
fpr_l2, tpr_l2, _ = roc_curve(y_test_arr, y_prob_l2)
roc_auc_l2 = auc(fpr_l2, tpr_l2)
fpr_l1, tpr_l1, _ = roc_curve(y_test_arr, y_prob_l1)
roc_auc_l1 = auc(fpr_l1, tpr_l1)

# plot ROC curves
plt.figure(figsize=(7,6))
plt.plot(fpr_l2, tpr_l2, label=f"L2 (AUC = {roc_auc_l2:.3f})")
plt.plot(fpr_l1, tpr_l1, label=f"L1 (AUC = {roc_auc_l1:.3f})")
plt.plot([0,1],[0,1],'--', linewidth=0.7)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: L2 vs L1")
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', linewidth=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "roc_curve_l2_l1.png", dpi=150)
plt.close()

# Confusion matrices (display + save)
cm_l2 = confusion_matrix(y_test_arr, y_pred_l2)
cm_l1 = confusion_matrix(y_test_arr, y_pred_l1)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_l2)
fig, ax = plt.subplots(figsize=(5,4))
disp.plot(ax=ax)
ax.set_title("Confusion Matrix - L2")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_l2.png", dpi=150)
plt.close()

disp = ConfusionMatrixDisplay(confusion_matrix=cm_l1)
fig, ax = plt.subplots(figsize=(5,4))
disp.plot(ax=ax)
ax.set_title("Confusion Matrix - L1")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_l1.png", dpi=150)
plt.close()

# Compute extra losses/metrics
logloss_l2 = log_loss(y_test_arr, y_prob_l2, labels=[0,1])
logloss_l1 = log_loss(y_test_arr, y_prob_l1, labels=[0,1])
brier_l2 = brier_score_loss(y_test_arr, y_prob_l2)
brier_l1 = brier_score_loss(y_test_arr, y_prob_l1)

# R2 and RMSE on probabilities vs true binary labels (overall)
r2_l2 = r2_score(y_test_arr, y_prob_l2)
r2_l1 = r2_score(y_test_arr, y_prob_l1)

# --- RMSE fix for older sklearn versions (mean_squared_error may not support squared=False) ---
mse_l2 = mean_squared_error(y_test_arr, y_prob_l2)
mse_l1 = mean_squared_error(y_test_arr, y_prob_l1)
rmse_l2 = float(np.sqrt(mse_l2))
rmse_l1 = float(np.sqrt(mse_l1))

# R2 and RMSE for top-20 predictions by predicted prob (if test set has >=20 rows)
def top_k_metrics(y_true, y_prob, k=20):
    n = len(y_prob)
    if n == 0:
        return {"r2_topk": None, "rmse_topk": None}
    idx = np.argsort(-y_prob)  # descending
    k2 = min(k, n)
    top_idx = idx[:k2]
    # r2
    r2_top = float(r2_score(y_true[top_idx], y_prob[top_idx])) if k2>0 else None
    # rmse via mse -> sqrt
    mse_top = mean_squared_error(y_true[top_idx], y_prob[top_idx])
    rmse_top = float(np.sqrt(mse_top)) if k2>0 else None
    return {
        "r2_topk": r2_top,
        "rmse_topk": rmse_top
    }

top20_l2 = top_k_metrics(y_test_arr, y_prob_l2, k=20)
top20_l1 = top_k_metrics(y_test_arr, y_prob_l1, k=20)

# Save these extended metrics
extended_metrics = {
    "logloss_l2": float(logloss_l2),
    "logloss_l1": float(logloss_l1),
    "brier_l2": float(brier_l2),
    "brier_l1": float(brier_l1),
    "r2_l2": float(r2_l2),
    "r2_l1": float(r2_l1),
    "rmse_l2": float(rmse_l2),
    "rmse_l1": float(rmse_l1),
    "top20_l2": top20_l2,
    "top20_l1": top20_l1
}
with open(OUT_DIR / "extended_metrics.json", "w") as f:
    json.dump(extended_metrics, f, indent=2)

# Bar chart comparing common metrics
labels = ["Accuracy", "F1", "Precision", "Recall", "ROC AUC", "PR AUC", "Brier", "LogLoss"]
vals_l2 = [
    metrics_l2["accuracy"],
    metrics_l2["f1"],
    metrics_l2["precision"],
    metrics_l2["recall"],
    metrics_l2["roc_auc"],
    metrics_l2["pr_auc"],
    brier_l2,
    logloss_l2
]
vals_l1 = [
    metrics_l1["accuracy"],
    metrics_l1["f1"],
    metrics_l1["precision"],
    metrics_l1["recall"],
    metrics_l1["roc_auc"],
    metrics_l1["pr_auc"],
    brier_l1,
    logloss_l1
]

x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x - width/2, vals_l2, width, label='L2')
ax.bar(x + width/2, vals_l1, width, label='L1')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_title("Model Comparison on Key Metrics")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "metrics_comparison_bar.png", dpi=150)
plt.close()

# Scatter plot: predicted probability vs true label (jittered) to visualize calibration
fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(y_prob_l2, y_test_arr + (np.random.randn(len(y_test_arr)) * 0.02), alpha=0.7, label='L2', s=20)
ax.scatter(y_prob_l1, y_test_arr + (np.random.randn(len(y_test_arr)) * 0.02), alpha=0.7, label='L1', s=20)
ax.set_xlabel("Predicted probability")
ax.set_ylabel("True label (jittered)")
ax.set_title("Predicted prob vs true label (jitter)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "prob_vs_true_scatter.png", dpi=150)
plt.close()

# Save R2/RMSE summary CSV
pd.DataFrame({
    "model": ["L2", "L1"],
    "r2": [r2_l2, r2_l1],
    "rmse": [rmse_l2, rmse_l1],
    "r2_top20": [top20_l2["r2_topk"], top20_l1["r2_topk"]],
    "rmse_top20": [top20_l2["rmse_topk"], top20_l1["rmse_topk"]]
}).to_csv(OUT_DIR / "r2_rmse_summary.csv", index=False)

print("Plots and extended metrics saved to:", OUT_DIR)


In [ ]:
#!/usr/bin/env python3
"""
Fast developability training — Stage-1 only (LightGBM RandomizedSearch)
- Stage-2 REMOVED (no GridSearch / early stopping) to avoid Jupyter flooding issues.
- n_iter = 6 for now (short debugging mode). Can be increased later.
- Outputs: metrics, cv_results, predictions, plots, feature importances, threshold sweep.
"""

import os
import json
import joblib
from pathlib import Path
from datetime import datetime, UTC
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# sklearn imports
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, learning_curve
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, matthews_corrcoef, brier_score_loss,
    precision_recall_curve
)
from sklearn.calibration import CalibratedClassifierCV
import shap
from sklearn.metrics import mean_squared_error, r2_score
# LightGBM
import lightgbm as lgb

# quiet warnings
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_style("whitegrid")

# ---------------- CONFIG ----------------
INPUT_CSV = "/content/drive/MyDrive/Aganitha/Model/RFF/AB_allebmpcdesc_clean_top15_afterlogloasso.csv"
BASE_OUTDIR = Path("/content/drive/MyDrive/Aganitha/Model/RFF/all_output")
RUN_PREFIX = "run"
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_TOP_FEATURES = 25

# weights / thresholds
WEIGHTS = {
    "normalized_titer_productionbatch1_avg": 0.4,
    "titer_productionbatch1_avg": 0.4,
    "purity_%lc+hc_avg": 1.0,
    "sec_%monomer_avg": 1.2,
    "tonset_nanodsf_avg": 1.0,
    "tm1_nanodsf_avg": 1.0,
    "tm2_nanodsf_avg": 1.0,
    "smac_rt_avg": 1.2,
    "hic_rt_avg": 1.2,
    "hac_rt_avg": 1.2,
    "acsins_dLmax_ph7.4_avg": 1.2,
    "acsins_dLmax_ph6.0_avg": 1.2,
    "polyreactivity_prscore_cho_avg": 1.2,
    "polyreactivity_prscore_ova_avg": 1.2,
}

DEVELOPABILITY_THRESHOLD = 0.95

EXCLUDE_COLS = [
    "antibody_id", "antibody_name",
    "hc_subtype", "lc_subtype",
    "highest_clinical_trial_asof_feb2025",
    "est_status_asof_feb2025",
    "SEQ_VH_clean", "SEQ_VL_clean",
    "SEQ_VHVL_concat_linker",
    "SEQ_HC_full_clean", "SEQ_LC_full_clean",

    # >>> YOUR 12 ASSAY COLUMNS HERE <<<

    "normalized_titer_productionbatch1_avg",
    "purity_%lc+hc_avg",
    "tonset_nanodsf_avg",
    "tm1_nanodsf_avg",
    "tm2_nanodsf_avg",
    "sec_%monomer_avg",
    "smac_rt_avg",
    "hic_rt_avg",
    "acsins_dLmax_ph7.4_avg",
    "acsins_dLmax_ph6.0_avg",
    "polyreactivity_prscore_cho_avg",
    "polyreactivity_prscore_ova_avg",
]

THRESHOLDS = {
    "normalized_titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "purity_%lc+hc_avg": {"high": 95.0, "medium": 90.0, "direction": "high"},
    "sec_%monomer_avg": {"high": 97.0, "medium": 95.0, "direction": "high"},
    "tonset_nanodsf_avg": {"high": 60.0, "medium": 55.0, "direction": "high"},
    "tm1_nanodsf_avg": {"high": 63.0, "medium": 60.0, "direction": "high"},
    "tm2_nanodsf_avg": {"high": 75.0, "medium": 70.0, "direction": "high"},
    "smac_rt_avg": {"high": 12.8, "medium": 15.0, "direction": "low"},
    "hic_rt_avg": {"high": 11.7, "medium": 14.0, "direction": "low"},
    "hac_rt_avg": {"high": 12.0, "medium": 15.0, "direction": "low"},
    "acsins_dLmax_ph7.4_avg": {"high": 11.8, "medium": 20.0, "direction": "low"},
    "acsins_dLmax_ph6.0_avg": {"high": 12.0, "medium": 20.0, "direction": "low"},
    "polyreactivity_prscore_cho_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
    "polyreactivity_prscore_ova_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
}

ASSAYS = list(THRESHOLDS.keys())
SOFT_ASSAYS = ["normalized_titer_productionbatch1_avg", "titer_productionbatch1_avg"]

# ---------------- helpers ----------------
def next_run_folder(base_dir: Path, prefix="run"):
    """Ensure run001, run002, run003… correct numeric order."""
    base_dir.mkdir(parents=True, exist_ok=True)
    nums = []
    for p in base_dir.iterdir():
        if p.is_dir() and p.name.startswith(prefix):
            suffix = p.name.replace(prefix, "")
            if suffix.isdigit():
                nums.append(int(suffix))
    new_num = max(nums) + 1 if nums else 1
    return base_dir / f"{prefix}{new_num}"

def safe_savefig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def tier_for_value(val, thr):
    if pd.isna(val):
        return np.nan
    if thr["direction"] == "high":
        return "High" if val >= thr["high"] else ("Medium" if val >= thr["medium"] else "Low")
    else:
        return "High" if val <= thr["high"] else ("Medium" if val <= thr["medium"] else "Low")

# ---------------- LOAD DATA ----------------
if not Path(INPUT_CSV).exists():
    raise FileNotFoundError(f"Missing: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)

# ---------------- TIERS & PASSES ----------------
for assay, thr in THRESHOLDS.items():
    if assay not in df.columns:
        df[assay] = np.nan
    df[f"{assay}_tier"] = df[assay].apply(lambda v: tier_for_value(v, thr))
    df[f"{assay}_pass"] = df[f"{assay}_tier"].map({"High": 1, "Medium": 1, "Low": 0})

# ---------------- WEIGHTED DEVELOPABILITY ----------------
pass_cols = [f"{a}_pass" for a in ASSAYS]

# --- SAFE WEIGHT VECTOR: default to 1.0 if key missing (won't override provided weights) ---
weight_vec = np.array([WEIGHTS.get(a, 1.0) for a in ASSAYS], float)

passes_matrix = df[pass_cols].to_numpy(float)
available_mask = ~np.isnan(passes_matrix)

weighted_pass = np.nansum(np.where(available_mask, passes_matrix * weight_vec, 0.0), axis=1)
available_weight = np.sum(np.where(available_mask, weight_vec, 0.0), axis=1)
weighted_fraction = np.where(available_weight > 0, weighted_pass / available_weight, np.nan)

df["weighted_pass_sum"] = weighted_pass
df["available_weight_sum"] = available_weight
df["developability_fraction"] = weighted_fraction
df["developability_index_unweighted"] = df[pass_cols].sum(axis=1, skipna=True)
df["developable_bin"] = (df["developability_fraction"] >= DEVELOPABILITY_THRESHOLD).astype(int)
df.loc[df["developability_fraction"].isna(), "developable_bin"] = np.nan

# ---------------- FEATURES FOR ML ----------------
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
tier_cols = [c for c in df.columns if c.endswith("_tier")]
pass_cols_actual = [c for c in df.columns if c.endswith("_pass")]

cols_to_remove = (
    set(tier_cols) |
    set(pass_cols_actual) |
    set(EXCLUDE_COLS) |
    set(ASSAYS) |
    {"developable_bin", "developability_fraction", "developability_index_unweighted",
     "weighted_pass_sum", "available_weight_sum"}
)

feature_cols = [c for c in numeric_cols if c not in cols_to_remove]
feat_missing_frac = df[feature_cols].isna().mean() if feature_cols else pd.Series(dtype=float)
feature_cols = [c for c in feature_cols if feat_missing_frac[c] <= 0.5]

print(f"LEAK-FREE FEATURE COUNT: {len(feature_cols)}")

imputer = SimpleImputer(strategy="median")
X_all = imputer.fit_transform(df[feature_cols].values)

y_series = df["developable_bin"]
has_label_mask = ~y_series.isna()
y_all = y_series[has_label_mask].astype(int).to_numpy()
X_all_labeled = X_all[has_label_mask.values, :]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all_labeled, y_all, df.index.values[has_label_mask.values],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)

# ---------------- RUN DIR ----------------
run_dir = next_run_folder(BASE_OUTDIR, RUN_PREFIX)
run_dir.mkdir(parents=True, exist_ok=True)

config = {
    "input_file": INPUT_CSV,
    "date": "Omitted by design",
    "thresholds": THRESHOLDS,
    "weights": WEIGHTS,
    "developability_threshold_value": DEVELOPABILITY_THRESHOLD,
    "n_features": len(feature_cols),
    "feature_columns": feature_cols
}

with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

with open(run_dir / "run_config.txt", "w") as f:
    f.write("=== STAGE-1 ONLY: RANDOMIZED SEARCH (LightGBM) ===\n")
    f.write(f"Run Directory: {run_dir}\nDate (UTC): {datetime.now(UTC).isoformat()}\n")
    f.write(f"Features used: {len(feature_cols)}\n")

# ---------------- RANDOMIZED SEARCH ----------------
print("Starting Stage-1 RandomizedSearchCV (LightGBM)")

param_distributions_lgb = {
    "n_estimators": [200],
    "learning_rate": [0.1],
    "num_leaves": [16],
    "max_depth": [4],
    "min_child_samples": [10],
    "subsample": [0.6],
    "colsample_bytree": [0.6],
    "reg_alpha": [0.0],
    "reg_lambda": [0.1],
    "class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lgb_est = lgb.LGBMClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
    device="gpu",
    gpu_platform_id=0,
    gpu_device_id=0
)

random_search = RandomizedSearchCV(
    estimator=lgb_est,
    param_distributions=param_distributions_lgb,
    n_iter=25,
    scoring="average_precision",
    cv=cv,
    refit=True,
    random_state=RANDOM_STATE,
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

# calibrated model
base_model = random_search.best_estimator_
clf = CalibratedClassifierCV(base_model, cv=5, method="sigmoid")
clf.fit(X_train, y_train)

best_params = random_search.best_params_

# update config
config["algorithm"] = "LightGBM"
config["best_hyperparameters_lgb"] = best_params
config["best_cv_score_stage1_average_precision"] = float(random_search.best_score_)
with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

pd.DataFrame(random_search.cv_results_).to_csv(run_dir / "cv_results_stage1.csv", index=False)

joblib.dump({"model": clf, "features": feature_cols, "imputer": imputer},
            str(run_dir / "lgb_stage1_model.joblib"))

print("Stage-1 complete. Best params:", best_params)
print("Best CV (average_precision):", random_search.best_score_)

# ---------------- METRICS & PREDICTIONS ----------------
y_proba = clf.predict_proba(X_test)[:, 1]

try:
    shap_sample = X_test if X_test.shape[0] <= 500 else X_test[np.random.RandomState(RANDOM_STATE).choice(X_test.shape[0], 500, replace=False)]
    explainer = shap.TreeExplainer(base_model)
    shap_values = explainer.shap_values(shap_sample)

    if isinstance(shap_values, list) and len(shap_values) > 1:
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    plt.figure(figsize=(8,6))
    shap.summary_plot(shap_vals, shap_sample, feature_names=feature_cols, show=False)
    safe_savefig(plt.gcf(), run_dir / "shap_summary.png")

    mean_abs_shap = np.abs(shap_vals).mean(axis=0)
    shap_df = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs_shap})
    shap_df = shap_df.sort_values("mean_abs_shap", ascending=False)
    shap_df.to_csv(run_dir / "shap_mean_abs_values.csv", index=False)

except Exception as e_shap:
    print("SHAP computation failed:", str(e_shap))

# 2) RMSE and R2 on probabilities
try:
    rmse_prob = float(np.sqrt(mean_squared_error(y_test, y_proba)))
    r2_prob = float(r2_score(y_test, y_proba))
except:
    rmse_prob, r2_prob = float("nan"), float("nan")

# will merge into main metrics object below
extra_reg_metrics = {
    "rmse_prob_vs_label": rmse_prob,
    "r2_prob_vs_label": r2_prob
}
# PR curve sweep BEFORE choosing cutoff
precision_arr, recall_arr, pr_thresholds = precision_recall_curve(y_test, y_proba)
f1_arr = [0.0 if (p + r) == 0 else 2 * p * r / (p + r)
          for p, r in zip(precision_arr[:-1], recall_arr[:-1])]

pr_sweep_df = pd.DataFrame({
    "threshold": pr_thresholds,
    "precision": precision_arr[:-1],
    "recall": recall_arr[:-1],
    "f1": f1_arr
})
pr_sweep_df.to_csv(run_dir / "threshold_sweep_prcurve.csv", index=False)

# choose operational cutoffs
MIN_PREC = 0.60
candidates = pr_sweep_df[pr_sweep_df["precision"] >= MIN_PREC]

if not candidates.empty:
    cutoff_recall = float(candidates.sort_values("recall", ascending=False).iloc[0]["threshold"])
else:
    cutoff_recall = float(pr_sweep_df.sort_values("recall", ascending=False).iloc[0]["threshold"])

cutoff_f1 = float(pr_sweep_df.sort_values("f1", ascending=False).iloc[0]["threshold"])

print("\nTop 5 thresholds (by recall):")
print(pr_sweep_df.sort_values("recall", ascending=False).head())
print("\nTop 5 thresholds (by F1):")
print(pr_sweep_df.sort_values("f1", ascending=False).head())

print(f"\nSelected cutoff_recall: {cutoff_recall:.4f}")
print(f"Selected cutoff_f1: {cutoff_f1:.4f}")

# apply chosen cutoff for main metrics
y_pred = (y_proba >= cutoff_recall).astype(int)

# save metrics
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
    "pr_auc": float(average_precision_score(y_test, y_proba)),
    "mcc": float(matthews_corrcoef(y_test, y_pred)),
    "brier_loss": float(brier_score_loss(y_test, y_proba))
}

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
metrics.update({"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})
# include extra reg metrics
metrics.update(extra_reg_metrics)

with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# predictions file
# create preds_df for test rows (fixing typo and ordering)
preds_df = df.loc[idx_test, ["antibody_id", "antibody_name"] + ASSAYS].copy().reset_index(drop=True)

# attach true/pred/prob for the test set
preds_df["developable_true"] = y_test
preds_df["developable_pred"] = y_pred
preds_df["developable_prob"] = y_proba

# attach developability summary columns for test rows
preds_df["developability_fraction"] = df.loc[idx_test, "developability_fraction"].values
preds_df["available_weight_sum"] = df.loc[idx_test, "available_weight_sum"].values
preds_df["weighted_pass_sum"] = df.loc[idx_test, "weighted_pass_sum"].values

# save the full processed dataframe (all 246 rows) once
df.to_csv("/content/drive/MyDrive/Aganitha/Model/RFF/all_246_antibodies_with_developability.csv", index=False)
print("Saved full dataset with developability labels: /content/drive/MyDrive/Aganitha/Model/RFF/all_246_antibodies_with_developability.csv")

# save the test-set predictions (as before)
preds_df.to_csv(run_dir / "predictions_test.csv", index=False)
print("Saved test-set predictions:", run_dir / "predictions_test.csv")


# ---------------- PLOTS ----------------

# ROC
if len(np.unique(y_test)) == 2:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "roc_curve.png")

# PR
if len(np.unique(y_test)) == 2:
    ap = average_precision_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(recall_arr, precision_arr, label=f"AP = {ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "pr_curve.png")

# Confusion Matrix
plt.figure(figsize=(4,3))
sns.heatmap(np.array([[tn, fp],[fn, tp]]), annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
safe_savefig(plt.gcf(), run_dir / "confusion_matrix.png")

# Calibration
if len(np.unique(y_test)) == 2:
    from sklearn.calibration import calibration_curve
    prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)
    plt.figure(figsize=(6,5))
    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel("Mean predicted probability"); plt.ylabel("Fraction of positives")
    plt.title("Calibration plot"); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "calibration.png")

# --- ADDED: ACCURACY vs PROBABILITY CUTOFF SWEEP ---
try:
    acc_thresholds = np.linspace(0.0, 1.0, 101)
    acc_vals = []
    prec_vals = []
    rec_vals = []
    f1_vals = []
    for thr in acc_thresholds:
        y_thr = (y_proba >= thr).astype(int)
        acc_vals.append(accuracy_score(y_test, y_thr))
        prec_vals.append(precision_score(y_test, y_thr, zero_division=0))
        rec_vals.append(recall_score(y_test, y_thr, zero_division=0))
        f1_vals.append(f1_score(y_test, y_thr, zero_division=0))
    acc_df = pd.DataFrame({
        "threshold": acc_thresholds,
        "accuracy": acc_vals,
        "precision": prec_vals,
        "recall": rec_vals,
        "f1": f1_vals
    })
    acc_df.to_csv(run_dir / "accuracy_vs_cutoff.csv", index=False)

    plt.figure(figsize=(8,5))
    plt.plot(acc_thresholds, acc_vals, label="accuracy")
    plt.plot(acc_thresholds, prec_vals, label="precision")
    plt.plot(acc_thresholds, rec_vals, label="recall")
    plt.plot(acc_thresholds, f1_vals, label="f1")
    plt.xlabel("Probability cutoff"); plt.ylabel("Metric"); plt.title("Accuracy/Precision/Recall/F1 vs cutoff")
    plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "accuracy_precision_recall_f1_vs_cutoff.png")
except Exception as e_acc:
    print("Accuracy vs cutoff sweep failed:", str(e_acc))

# --- ADDED: LEARNING CURVE (train/validation) ---
try:
    train_sizes = np.linspace(0.1, 1.0, 5)
    # use base_model (not calibrated) to get learning behaviour
    train_sizes_abs, train_scores, val_scores = earning_curve(
    estimator,
    X,
    y,
    *,
    train_sizes=None,
    cv=None,
    scoring=None,
    n_jobs=None,
    shuffle=False,
    random_state=None   ← only used IF shuffle=True
)

    train_scores_mean = train_scores.mean(axis=1)
    train_scores_std = train_scores.std(axis=1)
    val_scores_mean = val_scores.mean(axis=1)
    val_scores_std = val_scores.std(axis=1)

    lc_df = pd.DataFrame({
        "train_size": train_sizes_abs,
        "train_score_mean": train_scores_mean,
        "train_score_std": train_scores_std,
        "val_score_mean": val_scores_mean,
        "val_score_std": val_scores_std
    })
    lc_df.to_csv(run_dir / "learning_curve_accuracy.csv", index=False)

    plt.figure(figsize=(7,5))
    plt.fill_between(train_sizes_abs, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1)
    plt.fill_between(train_sizes_abs, val_scores_mean - val_scores_std, val_scores_mean + val_scores_std, alpha=0.1)
    plt.plot(train_sizes_abs, train_scores_mean, 'o-', label='Train score')
    plt.plot(train_sizes_abs, val_scores_mean, 'o-', label='Validation score')
    plt.xlabel("Training examples"); plt.ylabel("Accuracy"); plt.title("Learning curve (accuracy)")
    plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "learning_curve_accuracy.png")
except Exception as e_lc:
    print("Learning curve computation failed:", str(e_lc))

# --- ADDED: CLASS DISTRIBUTION PLOT ---
try:
    vc = df["developable_bin"].value_counts(dropna=False).sort_index()
    plt.figure(figsize=(5,4))
    sns.barplot(x=vc.index.astype(str), y=vc.values)
    plt.xlabel("developable_bin"); plt.ylabel("count"); plt.title("Class distribution (full df)")
    safe_savefig(plt.gcf(), run_dir / "class_distribution_full.png")
    vc.loc[~df.loc[has_label_mask, 'developable_bin'].isna()].to_frame().to_csv(run_dir / "class_distribution_counts.csv")
except Exception as e_cd:
    print("Class distribution plot failed:", str(e_cd))

# --- ADDED: FEATURE CORRELATION HEATMAP ---
try:
    if len(feature_cols) >= 2:
        corr = df[feature_cols].corr(method="pearson")
        plt.figure(figsize=(max(6, 0.25*len(feature_cols)), max(6, 0.25*len(feature_cols))))
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=False, cmap="vlag", center=0)
        plt.title("Feature correlation (pearson)")
        safe_savefig(plt.gcf(), run_dir / "feature_correlation_heatmap.png")
        corr.to_csv(run_dir / "feature_correlation_matrix.csv")
except Exception as e_corr:
    print("Feature correlation heatmap failed:", str(e_corr))
# --- END ADDED BLOCKS ---

# Feature Importances (horizontal barplot for readability)
fi = base_model.feature_importances_
fi_idx = np.argsort(fi)[::-1][:N_TOP_FEATURES]
top_features = [(feature_cols[i], float(fi[i])) for i in fi_idx]
fi_df = pd.DataFrame(top_features, columns=["feature", "importance"])
fi_df.to_csv(run_dir / "feature_importances_top.csv", index=False)

plt.figure(figsize=(8, max(4, 0.25*len(fi_idx))))
sns.barplot(y=[n for n,_ in top_features], x=[f for _,f in top_features], orient='h')
plt.xlabel("Importance"); plt.ylabel("Feature")
plt.title("Top feature importances (LGBM)")
safe_savefig(plt.gcf(), run_dir / "feature_importances_top.png")

# Diagnostics
diag = {
    "n_total_rows": int(df.shape[0]),
    "n_rows_with_label": int(has_label_mask.sum()),
    "n_features_used": len(feature_cols),
    "developability_threshold_used": DEVELOPABILITY_THRESHOLD,
}
with open(run_dir / "diagnostics.json", "w") as f:
    json.dump(diag, f, indent=2)

print(f"\nRUN COMPLETE → {run_dir}")
print("Stage-1 only run finished (Stage-2 omitted).")


In [ ]:
# ============================================================
#  LOGISTIC REGRESSION (L2 + L1) FULL PIPELINE + LINEAR SHAP
#  RUN-READY COLAB CODE FOR REPORT GENERATION
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss,
    r2_score, mean_squared_error
)

import shap
shap.initjs()

# ============================================================
#                USER CONFIGURATION
# ============================================================

CSV_PATH = "/content/drive/MyDrive/Aganitha/Model/RFF/all_246_antibodies_with_developability.csv"

BASE_DIR = Path("/content/drive/MyDrive/Aganitha/Model/RFF")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Auto-increment run folder
i = 1
while (BASE_DIR / f"run{i}").exists():
    i += 1
RUN_DIR = BASE_DIR / f"run{i}"
RUN_DIR.mkdir()

print(f"Saving everything to: {RUN_DIR}")

TOP15_FEATURES = [
    "aliphatic_index","aa_frac_D","aa_frac_E","PC_ESM_1","PC_ProtBerty_30",
    "PC_ESM_3","PC_ESM_16","PC_ESM_14","PC_ProtBerty_8","PC_ESM_48",
    "PC_ESM_23","PC_ESM_49","PC_ESM_34","PC_ESM_20","aa_frac_Q"
]

RANDOM_STATE = 42
TEST_SIZE = 0.20

# ============================================================
#                LOAD + PREPARE DATA
# ============================================================

df = pd.read_csv(CSV_PATH)
print("Full DF:", df.shape)

df = df[df["developable_bin"].notna()].copy()
X = df[TOP15_FEATURES]
y = df["developable_bin"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, " | Test:", X_test.shape)

# ============================================================
#               PIPELINES (L2 & L1)
# ============================================================

base_pipe = [
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
]

pipe_l2 = Pipeline(base_pipe + [
    ("clf", LogisticRegression(penalty="l2", C=0.1, solver="liblinear",
                               max_iter=5000, random_state=RANDOM_STATE))
])

pipe_l1 = Pipeline(base_pipe + [
    ("clf", LogisticRegression(penalty="l1", C=1.0, solver="liblinear",
                               max_iter=5000, random_state=RANDOM_STATE))
])

pipe_l2.fit(X_train, y_train)
pipe_l1.fit(X_train, y_train)

y_prob_l2 = pipe_l2.predict_proba(X_test)[:,1]
y_prob_l1 = pipe_l1.predict_proba(X_test)[:,1]

y_pred_l2 = (y_prob_l2 >= 0.5).astype(int)
y_pred_l1 = (y_prob_l1 >= 0.5).astype(int)

# ============================================================
#               METRIC DICTIONARY FUNCTION
# ============================================================

def compute_metrics(y_true, y_prob, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_prob))),
        "r2": float(r2_score(y_true, y_prob)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }

metrics_l2 = compute_metrics(y_test, y_prob_l2, y_pred_l2)
metrics_l1 = compute_metrics(y_test, y_prob_l1, y_pred_l1)

# Save metrics
json.dump(metrics_l2, open(RUN_DIR/"metrics_l2.json","w"), indent=2)
json.dump(metrics_l1, open(RUN_DIR/"metrics_l1.json","w"), indent=2)

# ============================================================
#                LINEAR SHAP (fast, exact)
# ============================================================

# Fit scaler separately for SHAP
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

explainer_l2 = shap.LinearExplainer(pipe_l2.named_steps["clf"], X_train_scaled)
shap_values_l2 = explainer_l2.shap_values(X_test_scaled)

# SHAP summary plot
plt.figure(figsize=(8, 6))
shap.summary_plot(shap_values_l2, X_test, show=False)
plt.title("SHAP Summary Plot (L2 Logistic)")
plt.tight_layout()
plt.savefig(RUN_DIR/"shap_l2_summary.png", dpi=350)
plt.close()

# ============================================================
#                COEFFICIENT BAR PLOTS
# ============================================================

def coef_plot(pipe, title, filename):
    coefs = pipe.named_steps["clf"].coef_.ravel()
    order = np.argsort(np.abs(coefs))[::-1]
    plt.figure(figsize=(7,5))
    plt.barh(np.array(TOP15_FEATURES)[order], coefs[order])
    plt.title(title)
    plt.tight_layout()
    plt.savefig(RUN_DIR/filename, dpi=350)
    plt.close()

coef_plot(pipe_l2, "L2 Coefficients", "coef_l2.png")
coef_plot(pipe_l1, "L1 Coefficients (LASSO)", "coef_l1.png")

# ============================================================
#              FEATURE MASK PLOT (L1)
# ============================================================

l1_mask = (pipe_l1.named_steps["clf"].coef_.ravel() != 0).astype(int)
plt.figure(figsize=(7,4))
plt.bar(TOP15_FEATURES, l1_mask)
plt.xticks(rotation=90)
plt.title("L1 Selected Features (Mask)")
plt.tight_layout()
plt.savefig(RUN_DIR/"l1_mask.png", dpi=350)
plt.close()

# ============================================================
#         ROC, PR, CALIBRATION, CONFUSION, PROB DIST
# ============================================================

from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, precision_recall_curve

# ROC
fpr, tpr, _ = roc_curve(y_test, y_prob_l2)
plt.plot(fpr, tpr)
plt.title("ROC Curve (L2)")
plt.savefig(RUN_DIR/"roc_l2.png", dpi=350)
plt.close()

# PR
prec, rec, _ = precision_recall_curve(y_test, y_prob_l2)
plt.plot(rec, prec)
plt.title("PR Curve (L2)")
plt.savefig(RUN_DIR/"pr_l2.png", dpi=350)
plt.close()

# Calibration
prob_true, prob_pred = calibration_curve(y_test, y_prob_l2, n_bins=5)
plt.plot(prob_pred, prob_true, marker="o")
plt.title("Calibration Curve (L2)")
plt.savefig(RUN_DIR/"calibration_l2.png", dpi=350)
plt.close()

# Confusion matrix heatmap
# Confusion matrix heatmap (L2)
cm = confusion_matrix(y_test, y_pred_l2)

plt.figure(figsize=(4,4))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"],
            yticklabels=["True 0", "True 1"])
plt.title("Confusion Matrix (L2)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.savefig(RUN_DIR / "cm_l2.png", dpi=350)
plt.close()


# Probability distribution
plt.hist(y_prob_l2[y_test==0], bins=15, alpha=0.7, label="Class 0")
plt.hist(y_prob_l2[y_test==1], bins=15, alpha=0.7, label="Class 1")
plt.legend()
plt.title("Probability Distribution (L2)")
plt.savefig(RUN_DIR/"prob_dist_l2.png", dpi=350)
plt.close()

# ============================================================
#                  LEARNING CURVE
# ============================================================

train_sizes, train_scores, val_scores = learning_curve(
    pipe_l2, X, y, cv=5, train_sizes=np.linspace(0.2,1.0,5)
)

plt.figure(figsize=(6,4))
plt.plot(train_sizes, train_scores.mean(axis=1), label="Train")
plt.plot(train_sizes, val_scores.mean(axis=1), label="Validation")
plt.legend()
plt.title("Learning Curve (L2)")
plt.savefig(RUN_DIR/"learning_curve_l2.png", dpi=350)
plt.close()

# ============================================================
#                      SAVE SUMMARY
# ============================================================

summary = {
    "run_folder": str(RUN_DIR),
    "l2_metrics": metrics_l2,
    "l1_metrics": metrics_l1,
    "features": TOP15_FEATURES,
}
json.dump(summary, open(RUN_DIR/"summary.json","w"), indent=2)

print("Done. All outputs saved in:", RUN_DIR)


In [ ]:
# Robust Evaluation: repeated CV + OOF + bootstrap CI
# Paste this into a new Colab cell and run (I did not run it for you).

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    accuracy_score, precision_score, recall_score, confusion_matrix
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import json, time, sys

# ---------- USER SETTINGS ----------
CSV_PATH = "/content/drive/MyDrive/Aganitha/Model/RFF/all_246_antibodies_with_developability.csv"
OUT_DIR = Path("/content/drive/MyDrive/Aganitha/Model/RFF/robust_eval")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COL = "developable_bin"   # label column in your file
FEATURES = [
    "aa_frac_D","PC_ESM_14","PC_ProtBerty_30","aa_frac_Q","PC_ESM_49",
    "PC_ESM_20","PC_ESM_3","PC_ESM_16","aa_frac_E","PC_ESM_48",
    "PC_ProtBerty_8","PC_ESM_23","PC_ESM_1","PC_ESM_34","aliphatic_index"
]

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 5
BOOTSTRAP_N = 1000   # set to 500 if you want faster
CI_LOW = 2.5
CI_HIGH = 97.5

# Choose which models to evaluate: any subset of ["logreg_l2","logreg_l1","lgb"]
models_to_run = ["logreg_l2", "lgb"]  # change as desired

# ---------- helper functions ----------
def compute_metrics(y_true, y_prob, cutoff=0.5):
    y_pred = (y_prob >= cutoff).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true))==2 else float("nan"),
        "pr_auc": float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true))==2 else float("nan"),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
    }

def bootstrap_ci(metric_fn, y_true, y_score, n_boot=1000, seed=0, ci_low=CI_LOW, ci_high=CI_HIGH):
    rng = np.random.RandomState(seed)
    stats = []
    n = len(y_true)
    idx = np.arange(n)
    for _ in range(n_boot):
        sample_idx = rng.choice(idx, size=n, replace=True)
        try:
            stats.append(metric_fn(y_true[sample_idx], y_score[sample_idx]))
        except Exception:
            stats.append(np.nan)
    stats = np.array(stats, dtype=float)
    lo = float(np.nanpercentile(stats, ci_low))
    hi = float(np.nanpercentile(stats, ci_high))
    mean = float(np.nanmean(stats))
    return {"mean": mean, "ci_lower": lo, "ci_upper": hi}

# metric wrappers for bootstrap
def auc_metric(y_true, y_score): return roc_auc_score(y_true, y_score)
def ap_metric(y_true, y_score): return average_precision_score(y_true, y_score)
def f1_from_probs(y_true, y_score):
    # use 0.5 threshold for bootstrap estimate (fast/stable)
    y_pred = (y_score >= 0.5).astype(int)
    return f1_score(y_true, y_pred, zero_division=0)

# ---------- load data ----------
df = pd.read_csv(CSV_PATH)
print("Loaded:", df.shape)

# basic checks
if LABEL_COL not in df.columns:
    raise KeyError(f"Label column {LABEL_COL} not found in CSV")

missing_feats = [f for f in FEATURES if f not in df.columns]
if missing_feats:
    raise KeyError(f"Missing required feature columns: {missing_feats}")

# drop rows without labels (if any)
df = df[df[LABEL_COL].notna()].copy()
X = df[FEATURES].values
y = df[LABEL_COL].astype(int).values

# ---------- repeated CV OOF collection ----------
rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
oof_preds = {m: np.zeros(len(y), dtype=float) for m in models_to_run}
oof_counts = np.zeros(len(y), dtype=int)  # how many times each sample was in validation
per_fold_stats = {m: [] for m in models_to_run}

fold_i = 0
start_time = time.time()
for train_idx, val_idx in rskf.split(X, y):
    fold_i += 1
    print(f"Fold {fold_i} / {N_SPLITS*N_REPEATS}  (train {len(train_idx)} val {len(val_idx)})")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    for model_name in models_to_run:
        if model_name == "logreg_l2":
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(penalty="l2", C=0.1, solver="liblinear", max_iter=5000, random_state=RANDOM_STATE))
            ])
        elif model_name == "logreg_l1":
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(penalty="l1", C=1.0, solver="liblinear", max_iter=5000, random_state=RANDOM_STATE))
            ])
        elif model_name == "lgb":
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                # no scaler required for trees but keep pipeline shape
                ("clf", lgb.LGBMClassifier(
                    n_estimators=400, learning_rate=0.05, num_leaves=16,
                    max_depth=6, min_child_samples=20, subsample=0.8,
                    colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=0.1,
                    random_state=RANDOM_STATE, n_jobs=-1
                ))
            ])
        else:
            raise ValueError("Unknown model: " + model_name)

        pipe.fit(X_train, y_train)
        # some classifiers (like LGB) may not implement predict_proba for certain cases; assume they do
        prob = pipe.predict_proba(X_val)[:,1]
        # accumulate OOF probs (sum); we'll divide by counts later to average across repeats
        oof_preds[model_name][val_idx] += prob
        per_fold_stats[model_name].append(compute_metrics(y_val, prob, cutoff=0.5))

    oof_counts[val_idx] += 1

elapsed = time.time() - start_time
print(f"\nRepeated CV done in {elapsed:.1f}s. OOF coverage counts (unique values):", np.unique(oof_counts))

# average OOF probs across times each sample appeared in validation
for model_name in models_to_run:
    with np.errstate(invalid="ignore", divide="ignore"):
        oof_preds[model_name] = np.where(oof_counts>0, oof_preds[model_name] / oof_counts, np.nan)

# ---------- summarize per-fold metrics ----------
summary = {}
for model_name in models_to_run:
    df_stats = pd.DataFrame([
        {k: v for k, v in d.items() if k != "confusion_matrix"}
        for d in per_fold_stats[model_name]
    ])

    mean_stats = df_stats.mean().to_dict() if not df_stats.empty else {}
    std_stats = df_stats.std().to_dict() if not df_stats.empty else {}

    conf_mats = [d["confusion_matrix"] for d in per_fold_stats[model_name]]

    summary[model_name] = {
        "per_fold_mean": {k: float(v) for k,v in mean_stats.items()},
        "per_fold_std": {k: float(v) for k,v in std_stats.items()},
        "n_folds": len(df_stats),
        "confusion_matrices": conf_mats
    }

print("\nPer-fold mean/std summary:")
print(json.dumps(summary, indent=2))

# ---------- bootstrap on OOF predictions (CI) ----------
boot_results = {}
for model_name in models_to_run:
    probs = oof_preds[model_name]
    mask = ~np.isnan(probs)
    if mask.sum() != len(y):
        print(f"Warning: some samples have NaN OOF probs for {model_name} (covered {mask.sum()} of {len(y)})")
    y_true_masked = y[mask]
    probs_masked = probs[mask]

    if len(y_true_masked) == 0:
        print(f"Skipping bootstrap for {model_name} because no OOF predictions are present.")
        continue

    print(f"\nRunning bootstrap (n={BOOTSTRAP_N}) for {model_name} on {mask.sum()} samples...")
    auc_ci = bootstrap_ci(auc_metric, y_true_masked, probs_masked, n_boot=BOOTSTRAP_N, seed=RANDOM_STATE)
    ap_ci = bootstrap_ci(ap_metric, y_true_masked, probs_masked, n_boot=BOOTSTRAP_N, seed=RANDOM_STATE+1)
    f1_ci = bootstrap_ci(f1_from_probs, y_true_masked, probs_masked, n_boot=BOOTSTRAP_N, seed=RANDOM_STATE+2)

    # point estimates on OOF
    point_metrics = compute_metrics(y_true_masked, probs_masked, cutoff=0.5)
    boot_results[model_name] = {
        "point_metrics_oof": point_metrics,
        "auc_bootstrap": auc_ci,
        "ap_bootstrap": ap_ci,
        "f1_bootstrap": f1_ci
    }
    print(f"{model_name} OOF point metrics:", point_metrics)
    print(f"{model_name} AUC {CI_LOW:.1f}%-{CI_HIGH:.1f}% CI: {auc_ci['ci_lower']:.3f} - {auc_ci['ci_upper']:.3f}")
    print(f"{model_name} AP  {CI_LOW:.1f}%-{CI_HIGH:.1f}% CI: {ap_ci['ci_lower']:.3f} - {ap_ci['ci_upper']:.3f}")
    print(f"{model_name} F1  {CI_LOW:.1f}%-{CI_HIGH:.1f}% CI: {f1_ci['ci_lower']:.3f} - {f1_ci['ci_upper']:.3f}")

# ---------- save results ----------
out = {
    "csv_path": CSV_PATH,
    "label_col": LABEL_COL,
    "features": FEATURES,
    "n_splits": N_SPLITS,
    "n_repeats": N_REPEATS,
    "bootstrap_n": BOOTSTRAP_N,
    "models": boot_results,
    "per_fold_summary": summary
}
with open(OUT_DIR / "robust_eval_summary.json", "w") as f:
    json.dump(out, f, indent=2)

print("\nSaved robust evaluation summary to:", OUT_DIR / "robust_eval_summary.json")




```
# This is formatted as code
```

saved lightgbm code

In [ ]:
# @title
#!/usr/bin/env python3
"""
Fast developability training — Stage-1 only (LightGBM RandomizedSearch)
- Stage-2 REMOVED (no GridSearch / early stopping) to avoid Jupyter flooding issues.
- n_iter = 6 for now (short debugging mode). Can be increased later.
- Outputs: metrics, cv_results, predictions, plots, feature importances, threshold sweep.
"""

import os
import json
import joblib
from pathlib import Path
from datetime import datetime, UTC
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# sklearn imports
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, learning_curve
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, matthews_corrcoef, brier_score_loss,
    precision_recall_curve
)
from sklearn.calibration import CalibratedClassifierCV
import shap
from sklearn.metrics import mean_squared_error, r2_score
# LightGBM
import lightgbm as lgb

# quiet warnings
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_style("whitegrid")

# ---------------- CONFIG ----------------
INPUT_CSV = "/content/drive/MyDrive/Aganitha/Model/RFF/descriptors_batch_all_merged_nonzero.csv"
BASE_OUTDIR = Path("/content/drive/MyDrive/Aganitha/Model/RFF/all_output")
RUN_PREFIX = "run"
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_TOP_FEATURES = 25

# weights / thresholds
#WEIGHTS = {
  #  "normalized_titer_productionbatch1_avg": 0.4,
  #  "titer_productionbatch1_avg": 0.4,
  #  "purity_%lc+hc_avg": 1.0,
  #  "sec_%monomer_avg": 1.2,
  #  "tonset_nanodsf_avg": 1.0,
  #  "tm1_nanodsf_avg": 1.0,
  #  "tm2_nanodsf_avg": 1.0,
  #  "smac_rt_avg": 1.2,
  #  "hic_rt_avg": 1.2,
  #  "hac_rt_avg": 1.2,
  #  "acsins_dLmax_ph7.4_avg": 1.2,
  #  "acsins_dLmax_ph6.0_avg": 1.2,
  #  "polyreactivity_prscore_cho_avg": 1.2,
 #   "polyreactivity_prscore_ova_avg": 1.2,
#}
WEIGHTS = {
    "normalized_titer_productionbatch1_avg": 1.2,
    "titer_productionbatch1_avg": 1.2,

    "purity_%lc+hc_avg": 1.3,
    "sec_%monomer_avg": 1.3,

    "tonset_nanodsf_avg": 1.5,
    "tm1_nanodsf_avg": 1.5,
    "tm2_nanodsf_avg": 1.5,

    "smac_rt_avg": 0.9,

    "hic_rt_avg": 1.0,
    "hac_rt_avg": 0.0,

    "acsins_dLmax_ph7.4_avg": 0.8,
    "acsins_dLmax_ph6.0_avg": 0.7,

    "polyreactivity_prscore_cho_avg": 0.6,
    "polyreactivity_prscore_ova_avg": 0.6,
}

DEVELOPABILITY_THRESHOLD = 0.95

EXCLUDE_COLS = [
    "antibody_id", "antibody_name",
    "hc_subtype", "lc_subtype",
    "highest_clinical_trial_asof_feb2025",
    "est_status_asof_feb2025",
    "SEQ_VH_clean", "SEQ_VL_clean",
    "SEQ_VHVL_concat_linker",
    "SEQ_HC_full_clean", "SEQ_LC_full_clean",

    # >>> YOUR 12 ASSAY COLUMNS HERE <<<

    #"normalized_titer_productionbatch1_avg",
    #"purity_%lc+hc_avg",
    #"tonset_nanodsf_avg",
    #"tm1_nanodsf_avg",
    #"tm2_nanodsf_avg",
    #"sec_%monomer_avg",
    #"smac_rt_avg",
    #"hic_rt_avg",
    #"acsins_dLmax_ph7.4_avg",
    #"acsins_dLmax_ph6.0_avg",
    #"polyreactivity_prscore_cho_avg",
    #"polyreactivity_prscore_ova_avg",
]

THRESHOLDS = {
    "normalized_titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "titer_productionbatch1_avg": {"high": 200.0, "medium": 100.0, "direction": "high"},
    "purity_%lc+hc_avg": {"high": 95.0, "medium": 90.0, "direction": "high"},
    "sec_%monomer_avg": {"high": 97.0, "medium": 95.0, "direction": "high"},
    "tonset_nanodsf_avg": {"high": 60.0, "medium": 55.0, "direction": "high"},
    "tm1_nanodsf_avg": {"high": 63.0, "medium": 60.0, "direction": "high"},
    "tm2_nanodsf_avg": {"high": 75.0, "medium": 70.0, "direction": "high"},
    "smac_rt_avg": {"high": 12.8, "medium": 15.0, "direction": "low"},
    "hic_rt_avg": {"high": 11.7, "medium": 14.0, "direction": "low"},
    "hac_rt_avg": {"high": 12.0, "medium": 15.0, "direction": "low"},
    "acsins_dLmax_ph7.4_avg": {"high": 11.8, "medium": 20.0, "direction": "low"},
    "acsins_dLmax_ph6.0_avg": {"high": 12.0, "medium": 20.0, "direction": "low"},
    "polyreactivity_prscore_cho_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
    "polyreactivity_prscore_ova_avg": {"high": 0.27, "medium": 0.35, "direction": "low"},
}

ASSAYS = list(THRESHOLDS.keys())
SOFT_ASSAYS = ["normalized_titer_productionbatch1_avg", "titer_productionbatch1_avg"]

# ---------------- helpers ----------------
def next_run_folder(base_dir: Path, prefix="run"):
    """Ensure run001, run002, run003… correct numeric order."""
    base_dir.mkdir(parents=True, exist_ok=True)
    nums = []
    for p in base_dir.iterdir():
        if p.is_dir() and p.name.startswith(prefix):
            suffix = p.name.replace(prefix, "")
            if suffix.isdigit():
                nums.append(int(suffix))
    new_num = max(nums) + 1 if nums else 1
    return base_dir / f"{prefix}{new_num}"

def safe_savefig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

def tier_for_value(val, thr):
    if pd.isna(val):
        return np.nan
    if thr["direction"] == "high":
        return "High" if val >= thr["high"] else ("Medium" if val >= thr["medium"] else "Low")
    else:
        return "High" if val <= thr["high"] else ("Medium" if val <= thr["medium"] else "Low")

# ---------------- LOAD DATA ----------------
if not Path(INPUT_CSV).exists():
    raise FileNotFoundError(f"Missing: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)

# ---------------- TIERS & PASSES ----------------
for assay, thr in THRESHOLDS.items():
    if assay not in df.columns:
        df[assay] = np.nan
    df[f"{assay}_tier"] = df[assay].apply(lambda v: tier_for_value(v, thr))
    df[f"{assay}_pass"] = df[f"{assay}_tier"].map({"High": 1, "Medium": 1, "Low": 0})

# ---------------- WEIGHTED DEVELOPABILITY ----------------
pass_cols = [f"{a}_pass" for a in ASSAYS]

# --- SAFE WEIGHT VECTOR: default to 1.0 if key missing (won't override provided weights) ---
weight_vec = np.array([WEIGHTS.get(a, 1.0) for a in ASSAYS], float)

passes_matrix = df[pass_cols].to_numpy(float)
available_mask = ~np.isnan(passes_matrix)

weighted_pass = np.nansum(np.where(available_mask, passes_matrix * weight_vec, 0.0), axis=1)
available_weight = np.sum(np.where(available_mask, weight_vec, 0.0), axis=1)
weighted_fraction = np.where(available_weight > 0, weighted_pass / available_weight, np.nan)

df["weighted_pass_sum"] = weighted_pass
df["available_weight_sum"] = available_weight
df["developability_fraction"] = weighted_fraction
df["developability_index_unweighted"] = df[pass_cols].sum(axis=1, skipna=True)
df["developable_bin"] = (df["developability_fraction"] >= DEVELOPABILITY_THRESHOLD).astype(int)
df.loc[df["developability_fraction"].isna(), "developable_bin"] = np.nan

# ---------------- FEATURES FOR ML ----------------
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
tier_cols = [c for c in df.columns if c.endswith("_tier")]
pass_cols_actual = [c for c in df.columns if c.endswith("_pass")]

cols_to_remove = (
    set(tier_cols) |
    set(pass_cols_actual) |
    set(EXCLUDE_COLS) |
    set(ASSAYS) |
    {"developable_bin", "developability_fraction", "developability_index_unweighted",
     "weighted_pass_sum", "available_weight_sum"}
)

feature_cols = [c for c in numeric_cols if c not in cols_to_remove]
feat_missing_frac = df[feature_cols].isna().mean() if feature_cols else pd.Series(dtype=float)
feature_cols = [c for c in feature_cols if feat_missing_frac[c] <= 0.5]

print(f"LEAK-FREE FEATURE COUNT: {len(feature_cols)}")

imputer = SimpleImputer(strategy="median")
X_all = imputer.fit_transform(df[feature_cols].values)

y_series = df["developable_bin"]
has_label_mask = ~y_series.isna()
y_all = y_series[has_label_mask].astype(int).to_numpy()
X_all_labeled = X_all[has_label_mask.values, :]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all_labeled, y_all, df.index.values[has_label_mask.values],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)

# ---------------- RUN DIR ----------------
run_dir = next_run_folder(BASE_OUTDIR, RUN_PREFIX)
run_dir.mkdir(parents=True, exist_ok=True)

config = {
    "input_file": INPUT_CSV,
    "date": "Omitted by design",
    "thresholds": THRESHOLDS,
    "weights": WEIGHTS,
    "developability_threshold_value": DEVELOPABILITY_THRESHOLD,
    "n_features": len(feature_cols),
    "feature_columns": feature_cols
}

with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

with open(run_dir / "run_config.txt", "w") as f:
    f.write("=== STAGE-1 ONLY: RANDOMIZED SEARCH (LightGBM) ===\n")
    f.write(f"Run Directory: {run_dir}\nDate (UTC): {datetime.now(UTC).isoformat()}\n")
    f.write(f"Features used: {len(feature_cols)}\n")

# ---------------- RANDOMIZED SEARCH ----------------
print("Starting Stage-1 RandomizedSearchCV (LightGBM)")

param_distributions_lgb = {
    "n_estimators": [200],
    "learning_rate": [0.1],
    "num_leaves": [16],
    "max_depth": [4],
    "min_child_samples": [10],
    "subsample": [0.6],
    "colsample_bytree": [0.6],
    "reg_alpha": [0.0],
    "reg_lambda": [0.1],
    "class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

try:
    lgb_est = lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
        device= "gpu",
        class_weight="balanced"
    )
    print("Using GPU for LightGBM")
except:
    lgb_est = lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
        device= "cpu",
        class_weight="balanced"
    )
    print("GPU unavailable → using CPU")

random_search = RandomizedSearchCV(
    estimator=lgb_est,
    param_distributions=param_distributions_lgb,
    n_iter=25,
    scoring="average_precision",
    cv=cv,
    refit=True,
    random_state=RANDOM_STATE,
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

# calibrated model
base_model = random_search.best_estimator_
clf = CalibratedClassifierCV(base_model, cv=5, method="sigmoid")
clf.fit(X_train, y_train)

best_params = random_search.best_params_

# update config
config["algorithm"] = "LightGBM"
config["best_hyperparameters_lgb"] = best_params
config["best_cv_score_stage1_average_precision"] = float(random_search.best_score_)
with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

pd.DataFrame(random_search.cv_results_).to_csv(run_dir / "cv_results_stage1.csv", index=False)

joblib.dump({"model": clf, "features": feature_cols, "imputer": imputer},
            str(run_dir / "lgb_stage1_model.joblib"))

print("Stage-1 complete. Best params:", best_params)
print("Best CV (average_precision):", random_search.best_score_)

# ---------------- METRICS & PREDICTIONS ----------------
y_proba = clf.predict_proba(X_test)[:, 1]

try:
    shap_sample = X_test if X_test.shape[0] <= 500 else X_test[np.random.RandomState(RANDOM_STATE).choice(X_test.shape[0], 500, replace=False)]
    explainer = shap.TreeExplainer(base_model)
    shap_values = explainer.shap_values(shap_sample)

    if isinstance(shap_values, list) and len(shap_values) > 1:
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    plt.figure(figsize=(8,6))
    shap.summary_plot(shap_vals, shap_sample, feature_names=feature_cols, show=False)
    safe_savefig(plt.gcf(), run_dir / "shap_summary.png")

    mean_abs_shap = np.abs(shap_vals).mean(axis=0)
    shap_df = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs_shap})
    shap_df = shap_df.sort_values("mean_abs_shap", ascending=False)
    shap_df.to_csv(run_dir / "shap_mean_abs_values.csv", index=False)

except Exception as e_shap:
    print("SHAP computation failed:", str(e_shap))

# 2) RMSE and R2 on probabilities
try:
    rmse_prob = float(np.sqrt(mean_squared_error(y_test, y_proba)))
    r2_prob = float(r2_score(y_test, y_proba))
except:
    rmse_prob, r2_prob = float("nan"), float("nan")

# will merge into main metrics object below
extra_reg_metrics = {
    "rmse_prob_vs_label": rmse_prob,
    "r2_prob_vs_label": r2_prob
}
# PR curve sweep BEFORE choosing cutoff
precision_arr, recall_arr, pr_thresholds = precision_recall_curve(y_test, y_proba)
f1_arr = [0.0 if (p + r) == 0 else 2 * p * r / (p + r)
          for p, r in zip(precision_arr[:-1], recall_arr[:-1])]

pr_sweep_df = pd.DataFrame({
    "threshold": pr_thresholds,
    "precision": precision_arr[:-1],
    "recall": recall_arr[:-1],
    "f1": f1_arr
})
pr_sweep_df.to_csv(run_dir / "threshold_sweep_prcurve.csv", index=False)

# choose operational cutoffs
MIN_PREC = 0.60
candidates = pr_sweep_df[pr_sweep_df["precision"] >= MIN_PREC]

if not candidates.empty:
    cutoff_recall = float(candidates.sort_values("recall", ascending=False).iloc[0]["threshold"])
else:
    cutoff_recall = float(pr_sweep_df.sort_values("recall", ascending=False).iloc[0]["threshold"])

cutoff_f1 = float(pr_sweep_df.sort_values("f1", ascending=False).iloc[0]["threshold"])

print("\nTop 5 thresholds (by recall):")
print(pr_sweep_df.sort_values("recall", ascending=False).head())
print("\nTop 5 thresholds (by F1):")
print(pr_sweep_df.sort_values("f1", ascending=False).head())

print(f"\nSelected cutoff_recall: {cutoff_recall:.4f}")
print(f"Selected cutoff_f1: {cutoff_f1:.4f}")

# apply chosen cutoff for main metrics
# = (y_proba >= cutoff_recall).astype(int)
y_pred = (y_proba >= cutoff_f1).astype(int)

# save metrics
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
    "pr_auc": float(average_precision_score(y_test, y_proba)),
    "mcc": float(matthews_corrcoef(y_test, y_pred)),
    "brier_loss": float(brier_score_loss(y_test, y_proba))
}

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
metrics.update({"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})
# include extra reg metrics
metrics.update(extra_reg_metrics)

with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# predictions file
# create preds_df for test rows (fixing typo and ordering)
preds_df = df.loc[idx_test, ["antibody_id", "antibody_name"] + ASSAYS].copy().reset_index(drop=True)

# attach true/pred/prob for the test set
preds_df["developable_true"] = y_test
preds_df["developable_pred"] = y_pred
preds_df["developable_prob"] = y_proba

# attach developability summary columns for test rows
preds_df["developability_fraction"] = df.loc[idx_test, "developability_fraction"].values
preds_df["available_weight_sum"] = df.loc[idx_test, "available_weight_sum"].values
preds_df["weighted_pass_sum"] = df.loc[idx_test, "weighted_pass_sum"].values

# save the full processed dataframe (all 246 rows) once
df.to_csv("/content/drive/MyDrive/Aganitha/Model/RFF/all_246_antibodies_with_developability.csv", index=False)
print("Saved full dataset with developability labels: /content/drive/MyDrive/Aganitha/Model/RFF/all_246_antibodies_with_developability.csv")

# save the test-set predictions (as before)
preds_df.to_csv(run_dir / "predictions_test.csv", index=False)
print("Saved test-set predictions:", run_dir / "predictions_test.csv")


# ---------------- PLOTS ----------------

# ROC
if len(np.unique(y_test)) == 2:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "roc_curve.png")

# PR
if len(np.unique(y_test)) == 2:
    ap = average_precision_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(recall_arr, precision_arr, label=f"AP = {ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "pr_curve.png")

# Confusion Matrix
plt.figure(figsize=(4,3))
sns.heatmap(np.array([[tn, fp],[fn, tp]]), annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
safe_savefig(plt.gcf(), run_dir / "confusion_matrix.png")

# Calibration
if len(np.unique(y_test)) == 2:
    from sklearn.calibration import calibration_curve
    prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)
    plt.figure(figsize=(6,5))
    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel("Mean predicted probability"); plt.ylabel("Fraction of positives")
    plt.title("Calibration plot"); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "calibration.png")

# --- ADDED: ACCURACY vs PROBABILITY CUTOFF SWEEP ---
try:
    acc_thresholds = np.linspace(0.0, 1.0, 101)
    acc_vals = []
    prec_vals = []
    rec_vals = []
    f1_vals = []
    for thr in acc_thresholds:
        y_thr = (y_proba >= thr).astype(int)
        acc_vals.append(accuracy_score(y_test, y_thr))
        prec_vals.append(precision_score(y_test, y_thr, zero_division=0))
        rec_vals.append(recall_score(y_test, y_thr, zero_division=0))
        f1_vals.append(f1_score(y_test, y_thr, zero_division=0))
    acc_df = pd.DataFrame({
        "threshold": acc_thresholds,
        "accuracy": acc_vals,
        "precision": prec_vals,
        "recall": rec_vals,
        "f1": f1_vals
    })
    acc_df.to_csv(run_dir / "accuracy_vs_cutoff.csv", index=False)

    plt.figure(figsize=(8,5))
    plt.plot(acc_thresholds, acc_vals, label="accuracy")
    plt.plot(acc_thresholds, prec_vals, label="precision")
    plt.plot(acc_thresholds, rec_vals, label="recall")
    plt.plot(acc_thresholds, f1_vals, label="f1")
    plt.xlabel("Probability cutoff"); plt.ylabel("Metric"); plt.title("Accuracy/Precision/Recall/F1 vs cutoff")
    plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "accuracy_precision_recall_f1_vs_cutoff.png")
except Exception as e_acc:
    print("Accuracy vs cutoff sweep failed:", str(e_acc))

# --- ADDED: LEARNING CURVE (train/validation) ---
try:
    train_sizes = np.linspace(0.1, 1.0, 5)
    # use base_model (not calibrated) to get learning behaviour
    train_sizes_abs, train_scores, val_scores = learning_curve(
        base_model, X_all_labeled, y_all, cv=cv, train_sizes=train_sizes,
        scoring="accuracy", n_jobs=-1, random_state=RANDOM_STATE
    )
    train_scores_mean = train_scores.mean(axis=1)
    train_scores_std = train_scores.std(axis=1)
    val_scores_mean = val_scores.mean(axis=1)
    val_scores_std = val_scores.std(axis=1)

    lc_df = pd.DataFrame({
        "train_size": train_sizes_abs,
        "train_score_mean": train_scores_mean,
        "train_score_std": train_scores_std,
        "val_score_mean": val_scores_mean,
        "val_score_std": val_scores_std
    })
    lc_df.to_csv(run_dir / "learning_curve_accuracy.csv", index=False)

    plt.figure(figsize=(7,5))
    plt.fill_between(train_sizes_abs, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1)
    plt.fill_between(train_sizes_abs, val_scores_mean - val_scores_std, val_scores_mean + val_scores_std, alpha=0.1)
    plt.plot(train_sizes_abs, train_scores_mean, 'o-', label='Train score')
    plt.plot(train_sizes_abs, val_scores_mean, 'o-', label='Validation score')
    plt.xlabel("Training examples"); plt.ylabel("Accuracy"); plt.title("Learning curve (accuracy)")
    plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "learning_curve_accuracy.png")
except Exception as e_lc:
    print("Learning curve computation failed:", str(e_lc))

# --- ADDED: CLASS DISTRIBUTION PLOT ---
try:
    vc = df["developable_bin"].value_counts(dropna=False).sort_index()
    plt.figure(figsize=(5,4))
    sns.barplot(x=vc.index.astype(str), y=vc.values)
    plt.xlabel("developable_bin"); plt.ylabel("count"); plt.title("Class distribution (full df)")
    safe_savefig(plt.gcf(), run_dir / "class_distribution_full.png")
    vc.loc[~df.loc[has_label_mask, 'developable_bin'].isna()].to_frame().to_csv(run_dir / "class_distribution_counts.csv")
except Exception as e_cd:
    print("Class distribution plot failed:", str(e_cd))

# --- ADDED: FEATURE CORRELATION HEATMAP ---
try:
    if len(feature_cols) >= 2:
        corr = df[feature_cols].corr(method="pearson")
        plt.figure(figsize=(max(6, 0.25*len(feature_cols)), max(6, 0.25*len(feature_cols))))
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=False, cmap="vlag", center=0)
        plt.title("Feature correlation (pearson)")
        safe_savefig(plt.gcf(), run_dir / "feature_correlation_heatmap.png")
        corr.to_csv(run_dir / "feature_correlation_matrix.csv")
except Exception as e_corr:
    print("Feature correlation heatmap failed:", str(e_corr))
# --- END ADDED BLOCKS ---

# Feature Importances (horizontal barplot for readability)
fi = base_model.feature_importances_
fi_idx = np.argsort(fi)[::-1][:N_TOP_FEATURES]
top_features = [(feature_cols[i], float(fi[i])) for i in fi_idx]
fi_df = pd.DataFrame(top_features, columns=["feature", "importance"])
fi_df.to_csv(run_dir / "feature_importances_top.csv", index=False)

plt.figure(figsize=(8, max(4, 0.25*len(fi_idx))))
sns.barplot(y=[n for n,_ in top_features], x=[f for _,f in top_features], orient='h')
plt.xlabel("Importance"); plt.ylabel("Feature")
plt.title("Top feature importances (LGBM)")
safe_savefig(plt.gcf(), run_dir / "feature_importances_top.png")

# Diagnostics
diag = {
    "n_total_rows": int(df.shape[0]),
    "n_rows_with_label": int(has_label_mask.sum()),
    "n_features_used": len(feature_cols),
    "developability_threshold_used": DEVELOPABILITY_THRESHOLD,
}
with open(run_dir / "diagnostics.json", "w") as f:
    json.dump(diag, f, indent=2)

print(f"\nRUN COMPLETE → {run_dir}")
print("Stage-1 only run finished (Stage-2 omitted).")


In [ ]:
#!/usr/bin/env python3
"""
Fast developability training — Stage-1 only (LightGBM RandomizedSearch)
- Stage-2 REMOVED (no GridSearch / early stopping) to avoid Jupyter flooding issues.
- n_iter = 6 for now (short debugging mode). Can be increased later.
- Outputs: metrics, cv_results, predictions, plots, feature importances, threshold sweep.

MODIFICATION: Medium tier removed from pass criteria. An assay only "passes"
if it meets the 'high' threshold. (tier_for_value now returns only "High"/"Low"
and the _pass mapping treats only "High" as pass; NaNs are preserved.)
"""

import os
import json
import joblib
from pathlib import Path
from datetime import datetime, UTC
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# sklearn imports
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, learning_curve
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, matthews_corrcoef, brier_score_loss,
    precision_recall_curve
)
from sklearn.calibration import CalibratedClassifierCV
import shap
from sklearn.metrics import mean_squared_error, r2_score
# LightGBM
import lightgbm as lgb

# quiet warnings
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_style("whitegrid")

# ---------------- CONFIG ----------------
INPUT_CSV = "/content/drive/MyDrive/Aganitha/Model/RFF/descriptors_batch_all_merged_nonzero.csv"
BASE_OUTDIR = Path("/content/drive/MyDrive/Aganitha/Model/RFF/all_output")
RUN_PREFIX = "run"
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_TOP_FEATURES = 25

 # weights / thresholds
WEIGHTS = {
    "normalized_titer_productionbatch1_avg": 0.8,
    "titer_productionbatch1_avg": 0.8,
    "purity_%lc+hc_avg": 1.2,
    "sec_%monomer_avg": 1.3,
    "tonset_nanodsf_avg": 1.5,
    "tm1_nanodsf_avg": 1.5,
    "tm2_nanodsf_avg": 1.5,
    "smac_rt_avg": 1.2,
    "hic_rt_avg": 1.2,
    "hac_rt_avg": 0.0,
    "acsins_dLmax_ph7.4_avg": 0.8,
    "acsins_dLmax_ph6.0_avg": 0.8,
    "polyreactivity_prscore_cho_avg": 1.0,
    "polyreactivity_prscore_ova_avg": 1.0,
}

DEVELOPABILITY_THRESHOLD = 0.82

EXCLUDE_COLS = [
    "antibody_id", "antibody_name",
    "hc_subtype", "lc_subtype",
    "highest_clinical_trial_asof_feb2025",
    "est_status_asof_feb2025",
    "SEQ_VH_clean", "SEQ_VL_clean",
    "SEQ_VHVL_concat_linker",
    "SEQ_HC_full_clean", "SEQ_LC_full_clean",
    # assay columns intentionally not listed here (they're handled separately)
]

THRESHOLDS = {
    "normalized_titer_productionbatch1_avg": {"high": 200.0, "medium": 150.0, "direction": "high"},
    "titer_productionbatch1_avg": {"high": 200.0, "medium": 150.0, "direction": "high"},
    "purity_%lc+hc_avg": {"high": 98.0, "medium": 96.0, "direction": "high"},
    "sec_%monomer_avg": {"high": 97.0, "medium": 95.0, "direction": "high"},
    "tonset_nanodsf_avg": {"high": 60.0, "medium": 60.0, "direction": "high"},
    "tm1_nanodsf_avg": {"high": 63.0, "medium": 75.0, "direction": "high"},
    "tm2_nanodsf_avg": {"high": 75.0, "medium": 80.0, "direction": "high"},
    "smac_rt_avg": {"high": 2.0, "medium": 3.0, "direction": "low"},
    "hic_rt_avg": {"high": 2.0, "medium": 3.5, "direction": "low"},
    "hac_rt_avg": {"high": 12.0, "medium": 15.0, "direction": "low"},
    "acsins_dLmax_ph7.4_avg": {"high": 11.8, "medium": 13.0, "direction": "low"},
    "acsins_dLmax_ph6.0_avg": {"high": 12.0, "medium": 13.0, "direction": "low"},
    "polyreactivity_prscore_cho_avg": {"high": 0.27, "medium": 0.29, "direction": "low"},
    "polyreactivity_prscore_ova_avg": {"high": 0.27, "medium": 0.29, "direction": "low"},
}

ASSAYS = list(THRESHOLDS.keys())
SOFT_ASSAYS = ["normalized_titer_productionbatch1_avg", "titer_productionbatch1_avg"]

# ---------------- helpers ----------------
def next_run_folder(base_dir: Path, prefix="run"):
    """Ensure run001, run002, run003… correct numeric order."""
    base_dir.mkdir(parents=True, exist_ok=True)
    nums = []
    for p in base_dir.iterdir():
        if p.is_dir() and p.name.startswith(prefix):
            suffix = p.name.replace(prefix, "")
            if suffix.isdigit():
                nums.append(int(suffix))
    new_num = max(nums) + 1 if nums else 1
    return base_dir / f"{prefix}{new_num}"

def safe_savefig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

# ---------------- MODIFIED: tier logic (NO MEDIUM) ----------------
def tier_for_value(val, thr):
    """
    Return only "High" or "Low" (or np.nan for missing).
    An assay is 'High' only if it meets the 'high' threshold.
    'medium' entries in THRESHOLDS are ignored for pass/tier.
    """
    if pd.isna(val):
        return np.nan

    if thr["direction"] == "high":
        return "High" if val >= thr["high"] else "Low"
    else:
        return "High" if val <= thr["high"] else "Low"

# ---------------- LOAD DATA ----------------
if not Path(INPUT_CSV).exists():
    raise FileNotFoundError(f"Missing: {INPUT_CSV}")

df = pd.read_csv(INPUT_CSV)

# ---------------- TIERS & PASSES ----------------
for assay, thr in THRESHOLDS.items():
    if assay not in df.columns:
        df[assay] = np.nan
    # compute tier (now only High/Low/NaN)
    df[f"{assay}_tier"] = df[assay].apply(lambda v: tier_for_value(v, thr))
    # compute pass: only High => 1, Low => 0, preserve NaN where tier is NaN
    df[f"{assay}_pass"] = df[f"{assay}_tier"].map({"High": 1, "Low": 0})
    df.loc[df[f"{assay}_tier"].isna(), f"{assay}_pass"] = np.nan

# ---------------- WEIGHTED DEVELOPABILITY ----------------
pass_cols = [f"{a}_pass" for a in ASSAYS]

# --- SAFE WEIGHT VECTOR: default to 1.0 if key missing (won't override provided weights) ---
weight_vec = np.array([WEIGHTS.get(a, 1.0) for a in ASSAYS], float)

passes_matrix = df[pass_cols].to_numpy(float)
available_mask = ~np.isnan(passes_matrix)

weighted_pass = np.nansum(np.where(available_mask, passes_matrix * weight_vec, 0.0), axis=1)
available_weight = np.sum(np.where(available_mask, weight_vec, 0.0), axis=1)
weighted_fraction = np.where(available_weight > 0, weighted_pass / available_weight, np.nan)

df["weighted_pass_sum"] = weighted_pass
df["available_weight_sum"] = available_weight
df["developability_fraction"] = weighted_fraction
df["developability_index_unweighted"] = df[pass_cols].sum(axis=1, skipna=True)
df["developable_bin"] = (df["developability_fraction"] >= DEVELOPABILITY_THRESHOLD).astype(int)
df.loc[df["developability_fraction"].isna(), "developable_bin"] = np.nan

# ---------------- FEATURES FOR ML ----------------
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
tier_cols = [c for c in df.columns if c.endswith("_tier")]
pass_cols_actual = [c for c in df.columns if c.endswith("_pass")]

cols_to_remove = (
    set(tier_cols) |
    set(pass_cols_actual) |
    set(EXCLUDE_COLS) |
    set(ASSAYS) |
    {"developable_bin", "developability_fraction", "developability_index_unweighted",
     "weighted_pass_sum", "available_weight_sum"}
)

feature_cols = [c for c in numeric_cols if c not in cols_to_remove]
feat_missing_frac = df[feature_cols].isna().mean() if feature_cols else pd.Series(dtype=float)
feature_cols = [c for c in feature_cols if feat_missing_frac[c] <= 0.5]

print(f"LEAK-FREE FEATURE COUNT: {len(feature_cols)}")

imputer = SimpleImputer(strategy="median")
X_all = imputer.fit_transform(df[feature_cols].values)

y_series = df["developable_bin"]
has_label_mask = ~y_series.isna()
y_all = y_series[has_label_mask].astype(int).to_numpy()
X_all_labeled = X_all[has_label_mask.values, :]

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_all_labeled, y_all, df.index.values[has_label_mask.values],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all
)

# ---------------- RUN DIR ----------------
run_dir = next_run_folder(BASE_OUTDIR, RUN_PREFIX)
run_dir.mkdir(parents=True, exist_ok=True)

config = {
    "input_file": INPUT_CSV,
    "date": "Omitted by design",
    "thresholds": THRESHOLDS,
    "weights": WEIGHTS,
    "developability_threshold_value": DEVELOPABILITY_THRESHOLD,
    "n_features": len(feature_cols),
    "feature_columns": feature_cols
}

with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

with open(run_dir / "run_config.txt", "w") as f:
    f.write("=== STAGE-1 ONLY: RANDOMIZED SEARCH (LightGBM) ===\n")
    f.write(f"Run Directory: {run_dir}\nDate (UTC): {datetime.now(UTC).isoformat()}\n")
    f.write(f"Features used: {len(feature_cols)}\n")

# ---------------- RANDOMIZED SEARCH ----------------
print("Starting Stage-1 RandomizedSearchCV (LightGBM)")

param_distributions_lgb = {
    "n_estimators": [200],
    "learning_rate": [0.1],
    "num_leaves": [16],
    "max_depth": [4],
    "min_child_samples": [10],
    "subsample": [0.6],
    "colsample_bytree": [0.6],
    "reg_alpha": [0.0],
    "reg_lambda": [0.1],
    "class_weight": [None, "balanced"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

try:
    lgb_est = lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
        device= "gpu",
        class_weight="balanced"
    )
    print("Using GPU for LightGBM")
except:
    lgb_est = lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
        device= "cpu",
        class_weight="balanced"
    )
    print("GPU unavailable → using CPU")

random_search = RandomizedSearchCV(
    estimator=lgb_est,
    param_distributions=param_distributions_lgb,
    n_iter=25,
    scoring="average_precision",
    cv=cv,
    refit=True,
    random_state=RANDOM_STATE,
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

# calibrated model
base_model = random_search.best_estimator_
clf = CalibratedClassifierCV(base_model, cv=5, method="sigmoid")
clf.fit(X_train, y_train)

best_params = random_search.best_params_

# update config
config["algorithm"] = "LightGBM"
config["best_hyperparameters_lgb"] = best_params
config["best_cv_score_stage1_average_precision"] = float(random_search.best_score_)
with open(run_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

pd.DataFrame(random_search.cv_results_).to_csv(run_dir / "cv_results_stage1.csv", index=False)

joblib.dump({"model": clf, "features": feature_cols, "imputer": imputer},
            str(run_dir / "lgb_stage1_model.joblib"))

print("Stage-1 complete. Best params:", best_params)
print("Best CV (average_precision):", random_search.best_score_)

# ---------------- METRICS & PREDICTIONS ----------------
y_proba = clf.predict_proba(X_test)[:, 1]

try:
    shap_sample = X_test if X_test.shape[0] <= 500 else X_test[np.random.RandomState(RANDOM_STATE).choice(X_test.shape[0], 500, replace=False)]
    explainer = shap.TreeExplainer(base_model)
    shap_values = explainer.shap_values(shap_sample)

    if isinstance(shap_values, list) and len(shap_values) > 1:
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    plt.figure(figsize=(8,6))
    shap.summary_plot(shap_vals, shap_sample, feature_names=feature_cols, show=False)
    safe_savefig(plt.gcf(), run_dir / "shap_summary.png")

    mean_abs_shap = np.abs(shap_vals).mean(axis=0)
    shap_df = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs_shap})
    shap_df = shap_df.sort_values("mean_abs_shap", ascending=False)
    shap_df.to_csv(run_dir / "shap_mean_abs_values.csv", index=False)

except Exception as e_shap:
    print("SHAP computation failed:", str(e_shap))

# 2) RMSE and R2 on probabilities
try:
    rmse_prob = float(np.sqrt(mean_squared_error(y_test, y_proba)))
    r2_prob = float(r2_score(y_test, y_proba))
except:
    rmse_prob, r2_prob = float("nan"), float("nan")

# will merge into main metrics object below
extra_reg_metrics = {
    "rmse_prob_vs_label": rmse_prob,
    "r2_prob_vs_label": r2_prob
}
# PR curve sweep BEFORE choosing cutoff
precision_arr, recall_arr, pr_thresholds = precision_recall_curve(y_test, y_proba)
f1_arr = [0.0 if (p + r) == 0 else 2 * p * r / (p + r)
          for p, r in zip(precision_arr[:-1], recall_arr[:-1])]

pr_sweep_df = pd.DataFrame({
    "threshold": pr_thresholds,
    "precision": precision_arr[:-1],
    "recall": recall_arr[:-1],
    "f1": f1_arr
})
pr_sweep_df.to_csv(run_dir / "threshold_sweep_prcurve.csv", index=False)

# choose operational cutoffs
MIN_PREC = 0.60
candidates = pr_sweep_df[pr_sweep_df["precision"] >= MIN_PREC]

if not candidates.empty:
    cutoff_recall = float(candidates.sort_values("recall", ascending=False).iloc[0]["threshold"])
else:
    cutoff_recall = float(pr_sweep_df.sort_values("recall", ascending=False).iloc[0]["threshold"])

cutoff_f1 = float(pr_sweep_df.sort_values("f1", ascending=False).iloc[0]["threshold"])

print("\nTop 5 thresholds (by recall):")
print(pr_sweep_df.sort_values("recall", ascending=False).head())
print("\nTop 5 thresholds (by F1):")
print(pr_sweep_df.sort_values("f1", ascending=False).head())

print(f"\nSelected cutoff_recall: {cutoff_recall:.4f}")
print(f"Selected cutoff_f1: {cutoff_f1:.4f}")

# apply chosen cutoff for main metrics
# = (y_proba >= cutoff_recall).astype(int)
y_pred = (y_proba >= cutoff_f1).astype(int)

# save metrics
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_proba)),
    "pr_auc": float(average_precision_score(y_test, y_proba)),
    "mcc": float(matthews_corrcoef(y_test, y_pred)),
    "brier_loss": float(brier_score_loss(y_test, y_proba))
}

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
metrics.update({"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})
# include extra reg metrics
metrics.update(extra_reg_metrics)

with open(run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# predictions file
# create preds_df for test rows (fixing typo and ordering)
preds_df = df.loc[idx_test, ["antibody_id", "antibody_name"] + ASSAYS].copy().reset_index(drop=True)

# attach true/pred/prob for the test set
preds_df["developable_true"] = y_test
preds_df["developable_pred"] = y_pred
preds_df["developable_prob"] = y_proba

# attach developability summary columns for test rows
preds_df["developability_fraction"] = df.loc[idx_test, "developability_fraction"].values
preds_df["available_weight_sum"] = df.loc[idx_test, "available_weight_sum"].values
preds_df["weighted_pass_sum"] = df.loc[idx_test, "weighted_pass_sum"].values

# save the full processed dataframe (all 246 rows) once
df.to_csv("/content/drive/MyDrive/Aganitha/Model/RFF/all_246_antibodies_with_developability.csv", index=False)
print("Saved full dataset with developability labels: /content/drive/MyDrive/Aganitha/Model/RFF/all_246_antibodies_with_developability.csv")

# save the test-set predictions (as before)
preds_df.to_csv(run_dir / "predictions_test.csv", index=False)
print("Saved test-set predictions:", run_dir / "predictions_test.csv")


# ---------------- PLOTS ----------------

# ROC
if len(np.unique(y_test)) == 2:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "roc_curve.png")

# PR
if len(np.unique(y_test)) == 2:
    ap = average_precision_score(y_test, y_proba)
    plt.figure(figsize=(6,5))
    plt.plot(recall_arr, precision_arr, label=f"AP = {ap:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve"); plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "pr_curve.png")

# Confusion Matrix
plt.figure(figsize=(4,3))
sns.heatmap(np.array([[tn, fp],[fn, tp]]), annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
safe_savefig(plt.gcf(), run_dir / "confusion_matrix.png")

# Calibration
if len(np.unique(y_test)) == 2:
    from sklearn.calibration import calibration_curve
    prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)
    plt.figure(figsize=(6,5))
    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0,1],[0,1],'--', color='gray')
    plt.xlabel("Mean predicted probability"); plt.ylabel("Fraction of positives")
    plt.title("Calibration plot"); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "calibration.png")

# --- ADDED: ACCURACY vs PROBABILITY CUTOFF SWEEP ---
try:
    acc_thresholds = np.linspace(0.0, 1.0, 101)
    acc_vals = []
    prec_vals = []
    rec_vals = []
    f1_vals = []
    for thr in acc_thresholds:
        y_thr = (y_proba >= thr).astype(int)
        acc_vals.append(accuracy_score(y_test, y_thr))
        prec_vals.append(precision_score(y_test, y_thr, zero_division=0))
        rec_vals.append(recall_score(y_test, y_thr, zero_division=0))
        f1_vals.append(f1_score(y_test, y_thr, zero_division=0))
    acc_df = pd.DataFrame({
        "threshold": acc_thresholds,
        "accuracy": acc_vals,
        "precision": prec_vals,
        "recall": rec_vals,
        "f1": f1_vals
    })
    acc_df.to_csv(run_dir / "accuracy_vs_cutoff.csv", index=False)

    plt.figure(figsize=(8,5))
    plt.plot(acc_thresholds, acc_vals, label="accuracy")
    plt.plot(acc_thresholds, prec_vals, label="precision")
    plt.plot(acc_thresholds, rec_vals, label="recall")
    plt.plot(acc_thresholds, f1_vals, label="f1")
    plt.xlabel("Probability cutoff"); plt.ylabel("Metric"); plt.title("Accuracy/Precision/Recall/F1 vs cutoff")
    plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "accuracy_precision_recall_f1_vs_cutoff.png")
except Exception as e_acc:
    print("Accuracy vs cutoff sweep failed:", str(e_acc))

# --- ADDED: LEARNING CURVE (train/validation) ---
try:
    train_sizes = np.linspace(0.1, 1.0, 5)
    # use base_model (not calibrated) to get learning behaviour
    train_sizes_abs, train_scores, val_scores = learning_curve(
        base_model, X_all_labeled, y_all, cv=cv, train_sizes=train_sizes,
        scoring="accuracy", n_jobs=-1, random_state=RANDOM_STATE
    )
    train_scores_mean = train_scores.mean(axis=1)
    train_scores_std = train_scores.std(axis=1)
    val_scores_mean = val_scores.mean(axis=1)
    val_scores_std = val_scores.std(axis=1)

    lc_df = pd.DataFrame({
        "train_size": train_sizes_abs,
        "train_score_mean": train_scores_mean,
        "train_score_std": train_scores_std,
        "val_score_mean": val_scores_mean,
        "val_score_std": val_scores_std
    })
    lc_df.to_csv(run_dir / "learning_curve_accuracy.csv", index=False)

    plt.figure(figsize=(7,5))
    plt.fill_between(train_sizes_abs, train_scores_mean - train_scores_std, train_scores_mean + train_scores_std, alpha=0.1)
    plt.fill_between(train_sizes_abs, val_scores_mean - val_scores_std, val_scores_mean + val_scores_std, alpha=0.1)
    plt.plot(train_sizes_abs, train_scores_mean, 'o-', label='Train score')
    plt.plot(train_sizes_abs, val_scores_mean, 'o-', label='Validation score')
    plt.xlabel("Training examples"); plt.ylabel("Accuracy"); plt.title("Learning curve (accuracy)")
    plt.legend(); plt.grid(True)
    safe_savefig(plt.gcf(), run_dir / "learning_curve_accuracy.png")
except Exception as e_lc:
    print("Learning curve computation failed:", str(e_lc))

# --- ADDED: CLASS DISTRIBUTION PLOT ---
try:
    vc = df["developable_bin"].value_counts(dropna=False).sort_index()
    plt.figure(figsize=(5,4))
    sns.barplot(x=vc.index.astype(str), y=vc.values)
    plt.xlabel("developable_bin"); plt.ylabel("count"); plt.title("Class distribution (full df)")
    safe_savefig(plt.gcf(), run_dir / "class_distribution_full.png")
    vc.loc[~df.loc[has_label_mask, 'developable_bin'].isna()].to_frame().to_csv(run_dir / "class_distribution_counts.csv")
except Exception as e_cd:
    print("Class distribution plot failed:", str(e_cd))

# --- ADDED: FEATURE CORRELATION HEATMAP ---
try:
    if len(feature_cols) >= 2:
        corr = df[feature_cols].corr(method="pearson")
        plt.figure(figsize=(max(6, 0.25*len(feature_cols)), max(6, 0.25*len(feature_cols))))
        mask = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, annot=False, cmap="vlag", center=0)
        plt.title("Feature correlation (pearson)")
        safe_savefig(plt.gcf(), run_dir / "feature_correlation_heatmap.png")
        corr.to_csv(run_dir / "feature_correlation_matrix.csv")
except Exception as e_corr:
    print("Feature correlation heatmap failed:", str(e_corr))
# --- END ADDED BLOCKS ---

# Feature Importances (horizontal barplot for readability)
fi = base_model.feature_importances_
fi_idx = np.argsort(fi)[::-1][:N_TOP_FEATURES]
top_features = [(feature_cols[i], float(fi[i])) for i in fi_idx]
fi_df = pd.DataFrame(top_features, columns=["feature", "importance"])
fi_df.to_csv(run_dir / "feature_importances_top.csv", index=False)

plt.figure(figsize=(8, max(4, 0.25*len(fi_idx))))
sns.barplot(y=[n for n,_ in top_features], x=[f for _,f in top_features], orient='h')
plt.xlabel("Importance"); plt.ylabel("Feature")
plt.title("Top feature importances (LGBM)")
safe_savefig(plt.gcf(), run_dir / "feature_importances_top.png")

# Diagnostics
diag = {
    "n_total_rows": int(df.shape[0]),
    "n_rows_with_label": int(has_label_mask.sum()),
    "n_features_used": len(feature_cols),
    "developability_threshold_used": DEVELOPABILITY_THRESHOLD,
}
with open(run_dir / "diagnostics.json", "w") as f:
    json.dump(diag, f, indent=2)

print(f"\nRUN COMPLETE → {run_dir}")
print("Stage-1 only run finished (Stage-2 omitted).")
